[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u6_aplicaciones.ipynb)

# Int. U6 · Aplicaciones a ingeniería
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 6 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

- 6.1 Área entre curvas
- 6.2 Volúmenes de sólidos de revolución
- 6.3 Longitud de arco
- 6.4 Centroides y momentos de inercia de área
- 6.5 Momentos de masa
- 6.6 De la aceleración a la posición
- 6.7 Trabajo, energía y eficiencia
- 6.8 Fuerza hidrostática y caudal
- 6.9 Carga, energía y valor eficaz de una señal
- 6.10 Costo acumulado

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría con un ejemplo exacto, una calculadora, sus casos de prueba y una celda **Contrasta**. Antes de ejecutar la celda Contrasta, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U6` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u6_aplicaciones/NN_*.py`; los fragmentos que el libro imprime, de `int/u6_aplicaciones/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que declara cada calculadora. Los ángulos de `sin`, `cos` y `tan` van en **radianes**. Cada resultado lleva su unidad: las secciones de física usan el SI (con prefijos como mA o µA cuando el dato viene así) y la sección 6.10 usa pesos. Para escribir funciones usa `x` como variable o la letra natural de la sección (`t`, `y`, `h`, `r` o `q`), `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)`, `log(...)` o `ln(...)` (natural), `abs(...)`, `sen` o `sin`, `cosh`, `pi` o `π`, `√x`, números como `2e5` y `escalon(t - 2)` para un escalón unitario que vale 0 antes de $t=2$ y 1 después.

## Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8" "scipy>=1.11"

import sys, math, re, signal, warnings
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from scipy.integrate import quad, IntegrationWarning
from scipy.optimize import brentq
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

import scipy
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)
CIFRAS = 6                                          # cifras significativas de las calculadoras

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log, "ln": sp.log,
              "pi": sp.pi, "sin": sp.sin, "sen": sp.sin, "cos": sp.cos, "tan": sp.tan,
              "atan": sp.atan, "arctan": sp.atan, "asin": sp.asin, "arcsen": sp.asin, "arcsin": sp.asin,
              "sinh": sp.sinh, "senh": sp.sinh, "cosh": sp.cosh, "escalon": sp.Heaviside, "Heaviside": sp.Heaviside,
              "E": sp.E, "e": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_OTRAS_VARIABLES = ("t", "y", "h", "r", "q", "s")     # se leen como x si son la única variable
_LITERAL = r"(?<![A-Za-z_\d.])(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?"   # 2, 0.5, .5, 2e5, 1.5e-3


def parsear(texto):
    """Convierte un texto en una expresión de sympy en x o explica qué salió mal.
    Acepta sen y ln, la letra e como número de Euler, π y √ (2πx, √x, √(x + 1)), |...| como valor absoluto,
    sin^2(x), números como 2e5, escalon(...) como escalón unitario y otra letra (t, y, h, r, q o s) en lugar de x
    si es la única variable."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    if "," in texto:
        raise ValueError("Usa punto decimal (1.5, no 1,5); las funciones de este notebook no llevan comas.")
    texto = texto.replace("·", "*").replace("−", "-").replace("escalón", "escalon")
    texto = texto.replace("π", " pi ")                                     # 2πx -> 2 pi x
    texto = re.sub(r"√\s*(\d+\.?\d*|[A-Za-z_]\w*|\([^()]*\))", r" sqrt(\1) ", texto)   # √x, √2, √(x + 1)
    texto = re.sub(r"\|([^|]+)\|", r"abs(\1)", texto)                      # |x - 1| -> abs(x - 1)
    texto = re.sub(r"\b(sin|sen|cos|tan)\s*\^\s*(\d+)\s*\(([^()]*)\)", r"(\1(\3))**\2", texto)   # sin^2(x)
    if "√" in texto:
        raise ValueError("Escribe la raíz como sqrt(...), por ejemplo sqrt(x + 1).")
    def _nombres(t):
        return set(re.findall(r"[A-Za-z_]\w*", re.sub(_LITERAL, " ", t)))   # sin los números: 2e5 no es un nombre
    nombres = _nombres(texto)
    otras = [v for v in _OTRAS_VARIABLES if v in nombres]
    if "x" not in nombres and len(otras) == 1:
        texto = re.sub(rf"(?<![A-Za-z_]){otras[0]}(?![A-Za-z_0-9])", "x", texto)
        nombres = _nombres(texto)
    local = {**_FUNCIONES, "x": x}
    desconocidos = sorted(nombres - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa una sola letra como variable (x, t, y, h, r o q), números "
                         "en lugar de letras para las constantes y, si hace falta, sqrt, exp, log o ln, abs, sin o sen, "
                         "cos, tan, cosh, escalon o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias, punto decimal "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión sin el signo = y solo con las funciones permitidas.")
    if expr.has(sp.oo, -sp.oo, sp.zoo, sp.nan):
        raise ValueError("La función no puede contener infinito.")
    if expr.free_symbols - {x}:
        raise ValueError("Usa una sola variable.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'12, 2, 11' -> [12.0, 2.0, 11.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 12, 2, 11")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Fuera de 1e-4 a 1e7 usa notación científica."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    if abs(v) >= 1e7 or abs(v) < 1e-4:
        m = (d.scaleb(-d.adjusted())).quantize(Decimal(1).scaleb(-n + 1), rounding=ROUND_HALF_UP)
        e = d.adjusted()
        if abs(m) >= 10:                     # 9.996e5 con 3 cifras pasa a 1.00e6
            m, e = (m / 10).quantize(Decimal(1).scaleb(-n + 1), rounding=ROUND_HALF_UP), e + 1
        return f"{m}e{e}"
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def coincide(mi, ref, n=3):
    """True si mi y ref coinciden a n cifras significativas (para las celdas Contrasta)."""
    mi, ref = float(mi), float(ref)
    return abs(mi - ref) <= 5 * 10**(-n) * max(abs(ref), 1e-12) or abs(mi - ref) < 1e-12


def a_numero(valor):
    """Número escrito por el alumno: 1.57, 'pi/2' o '3/2'. ValueError con un mensaje claro si no es un real finito."""
    if isinstance(valor, bool):
        raise ValueError("Escribe un número, no True ni False.")
    if isinstance(valor, (int, float)):
        if not math.isfinite(valor):
            raise ValueError("Escribe un número finito.")
        return float(valor)
    texto = str(valor).strip()
    if not texto:
        raise ValueError("Escribe tu resultado entre comillas, por ejemplo \"1.5\" o \"4/3\".")
    if re.fullmatch(r"[+-]?\d*,\d+", texto):
        raise ValueError(f"Usa punto decimal: escribe {texto.replace(',', '.')} en lugar de {texto}.")
    try:
        v = parsear(texto)
    except ValueError as err:
        if "infinito" in str(err):
            raise ValueError("Eso no es un número finito (¿dividiste entre cero?).") from None
        raise
    if v.free_symbols:
        raise ValueError("Escribe un número, sin x.")
    try:
        z = complex(sp.N(v))
    except (TypeError, ValueError):
        raise ValueError("Eso no es un número real.") from None
    if abs(z.imag) > 1e-12 or not math.isfinite(z.real):
        raise ValueError("Eso no es un número real finito.")
    return z.real


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


class _Tarda(BaseException):                 # BaseException: los "except Exception" internos de sympy no la atrapan
    pass


def con_limite(fn, segundos=15):
    """Ejecuta fn() con un límite de tiempo (en Colab); si se pasa, lanza ValueError con un mensaje claro."""
    if not hasattr(signal, "SIGALRM"):        # Windows: sin límite
        return fn()
    def _alarma(*_):
        raise _Tarda()
    anterior = signal.signal(signal.SIGALRM, _alarma)
    signal.alarm(segundos)
    try:
        return fn()
    except _Tarda:
        raise ValueError(f"sympy tardó más de {segundos} s con esta función; prueba una más sencilla.") from None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, anterior)


# Errores que una calculadora convierte en un mensaje para el alumno
ERRORES = (ValueError, TypeError, ArithmeticError)


def explica(err):
    """Mensaje en español para un error de cálculo."""
    if isinstance(err, ValueError):
        return str(err)
    if isinstance(err, OverflowError):
        return "los números crecen demasiado (desbordamiento): usa un intervalo más corto."
    if isinstance(err, ZeroDivisionError):
        return "apareció una división entre cero: revisa los datos."
    return "no pude convertir el resultado en un número real; prueba con otra función o con otro intervalo."


def numerica(f):
    """Versión numpy de f que siempre devuelve un arreglo de floats del tamaño de la entrada (nan donde no es real)."""
    g = sp.lambdify(x, sp.sympify(f), "numpy")
    def fn(t):
        t = np.asarray(t, dtype=float)
        with np.errstate(all="ignore"):
            y = np.asarray(g(t)) * np.ones_like(t)
        if np.iscomplexobj(y):
            y = np.where(np.abs(y.imag) < 1e-12, y.real, np.nan)
        return np.asarray(y, dtype=float)
    return fn


def para_graficar(f, xs):
    """Valores de f en xs, con nan donde no son finitos (matplotlib deja un hueco en lugar de fallar)."""
    ys = numerica(f)(xs)
    return np.where(np.isfinite(ys), ys, np.nan)


def exacto(v, nombre="el valor"):
    """Número de sympy, exacto si se puede (0.1 -> 1/10, 'pi/2' -> pi/2); ValueError si no es un real finito."""
    if isinstance(v, bool):
        raise ValueError(f"{nombre} debe ser un número.")
    if isinstance(v, sp.Basic):
        e = v
    elif isinstance(v, (int, np.integer)):
        e = sp.Integer(int(v))
    elif isinstance(v, (float, np.floating)):
        if not math.isfinite(float(v)):
            raise ValueError(f"{nombre} debe ser un número finito.")
        e = sp.Rational(repr(float(v)))
    else:
        try:
            e = parsear(str(v))
        except ValueError:
            raise ValueError(f"Escribe {nombre} como número: 3, 0.5, 1/2 o pi/2.") from None
    if getattr(e, "free_symbols", None):
        raise ValueError(f"{nombre} es un número: no lleva variable.")
    try:
        z = complex(sp.N(e))
    except (TypeError, ValueError):
        raise ValueError(f"{nombre} debe ser un número real.") from None
    if abs(z.imag) > 0 or not math.isfinite(z.real):
        raise ValueError(f"{nombre} debe ser un número real finito.")
    return e


def simplifica(v, segundos=5):
    """sympy.simplify con límite de tiempo; si tarda, deja el valor como está."""
    if not isinstance(v, sp.Basic) or isinstance(v, sp.Float):
        return v
    try:
        s_ = con_limite(lambda: sp.simplify(v), segundos)
    except ValueError:
        return v
    e_ = sp.expand(s_)                           # a veces la forma expandida es más corta: 1/100 - exp(-5)/100
    return e_ if len(str(e_)) < len(str(s_)) else s_


def _es_decimal_finito(v):
    """True si el racional v tiene expansión decimal finita (denominador con solo 2 y 5)."""
    q = int(v.q)
    for p_ in (2, 5):
        while q % p_ == 0:
            q //= p_
    return q == 1


def muestra(v, n=CIFRAS):
    """El valor redondeado a n cifras significativas. Si v es exacto y no es un decimal finito (16/3, 2*sqrt(2), pi),
    antepone su forma exacta: '16/3 ≈ 5.33333'. Un entero corto se escribe tal cual."""
    if isinstance(v, sp.Basic) and v.atoms(sp.Float):     # ya es decimal (viene de quad o de un cruce numérico)
        return cifras(v, n)
    if isinstance(v, sp.Integer) and len(str(abs(int(v)))) <= n:
        return str(v)
    if isinstance(v, sp.Rational) and _es_decimal_finito(v):
        return cifras(v, n)
    if isinstance(v, sp.Basic) and not isinstance(v, (sp.Float, sp.Integer)) and len(str(v)) < 40:
        return f"{v} ≈ {cifras(v, n)}"
    return cifras(v, n)


def malla_interior(a, b, n=401):
    """n puntos dentro de (a, b), sin los extremos (ahí puede haber una singularidad integrable)."""
    return np.linspace(float(a), float(b), n + 2)[1:-1]


def revisa_intervalo(f, a, b, nombre="f", var="x"):
    """ValueError si f no es real y continua dentro de (a, b). Decide sympy; con escalones, o si sympy no puede,
    se revisa en una malla de puntos. Los extremos se dejan libres: ahí la integral puede ser impropia y convergente.
    var es la letra que ve el alumno en los mensajes (t, h, r, q)."""
    lo, hi = sorted((a, b), key=float)
    f = sp.sympify(f)
    if not f.has(x):
        return
    if not f.has(sp.Heaviside, sp.Piecewise):
        try:
            dom = con_limite(lambda: sp.calculus.util.continuous_domain(f, x, sp.Interval.open(lo, hi)), 5)
        except (ValueError, NotImplementedError, TypeError):
            dom = None
        if dom is not None and dom != sp.Interval.open(lo, hi):
            fuera = sp.Interval.open(lo, hi) - dom
            if isinstance(fuera, sp.FiniteSet) and len(fuera) <= 4:
                donde = ", ".join(punto(p) for p in sorted(fuera, key=float))
                raise ValueError(f"{nombre} no es continua en {var} = {donde}: ahí hay una asíntota o un salto. "
                                 "Separa el intervalo o elige otro.")
    ys = numerica(f)(malla_interior(lo, hi))
    if np.any(np.isnan(ys)):
        raiz_impar = any(p.exp.is_Rational and p.exp.q % 2 == 1 and p.exp.q > 1 for p in f.atoms(sp.Pow))
        raise ValueError(f"{nombre} no está definida (o no es real) en todo el intervalo: revisa el dominio."
                         + (" Para sympy, una potencia como x^(2/3) o x^(1/3) no es real si la base es negativa; "
                            "usa un intervalo donde la base sea positiva." if raiz_impar else ""))
    if np.any(np.isinf(ys)):
        raise ValueError(f"{nombre} crece demasiado dentro del intervalo: sus valores pasan de 1e308 (desbordamiento). "
                         "Usa un intervalo más corto.")


# Integrales definidas: exactas con sympy si se puede; si no, numéricas con quad
_RARAS = (sp.Integral, sp.exp_polar, sp.hyper, sp.meijerg, sp.lowergamma, sp.uppergamma, sp.Piecewise,
          sp.polar_lift, sp.Min, sp.Max)


def _valor_exacto(valor):
    """float real del resultado de sympy, o None si no sirve: no es un número, es complejo, usa funciones que sympy
    no compara bien (exp_polar, hyper, ...) o tiene condiciones sin decidir."""
    if not isinstance(valor, sp.Expr) or valor.free_symbols or valor.has(*_RARAS) or valor.has(sp.I):
        return None
    if valor.atoms(sp.core.relational.Relational):
        return None
    try:
        z = complex(con_limite(lambda: sp.N(valor), 5))
    except (ValueError, TypeError, ArithmeticError):
        return None
    if not (math.isfinite(z.real) and math.isfinite(z.imag)) or abs(z.imag) > 1e-12 * max(1.0, abs(z.real)):
        return None
    return z.real


def _quad(fn, a, b):
    """∫_a^b por cuadratura, en tramos de longitud 50 como máximo (las oscilaciones largas no confunden a quad)."""
    tramos = min(2000, max(1, int(math.ceil(abs(b - a) / 50))))
    cortes = np.linspace(a, b, tramos + 1)
    total = 0.0
    with warnings.catch_warnings():
        warnings.simplefilter("error", IntegrationWarning)
        try:
            for p, q in zip(cortes, cortes[1:]):
                total += quad(lambda s: float(fn(s)), p, q, limit=200)[0]
        except (IntegrationWarning, ZeroDivisionError, OverflowError):
            raise ValueError("quad no alcanzó la precisión pedida: la integral puede diverger (si la función crece sin "
                             "límite cerca de algún punto) o la función oscila demasiado. Prueba con un intervalo más "
                             "corto.") from None
    if not math.isfinite(total):
        raise ValueError("la integral numérica no dio un número finito: revisa el dominio de la función.")
    return total


def integra(f, a, b, segundos=8, solo_numerica=False):
    """∫_a^b f(x) dx. Devuelve (valor, "exacta") si sympy la resuelve en el tiempo dado con una expresión útil;
    si no, (Float, "numérica") con quad. ValueError si diverge o si f no es real en el intervalo."""
    f = sp.sympify(f)
    a, b = exacto(a, "el extremo inferior"), exacto(b, "el extremo superior")
    if f.has(sp.Heaviside):                     # sympy integra mejor los escalones como funciones por tramos
        f = f.rewrite(sp.Piecewise)
    valor = None
    if not solo_numerica:
        try:
            valor = con_limite(lambda: sp.integrate(f, (x, a, b)), segundos)
        except (ValueError, NotImplementedError, TypeError, ArithmeticError):
            valor = None
    if isinstance(valor, sp.Expr) and (valor.has(sp.oo, -sp.oo, sp.zoo) and not valor.has(sp.nan)):
        raise ValueError("la integral diverge: la función crece sin límite en el intervalo.")
    z = _valor_exacto(valor) if valor is not None else None
    if z is not None:
        if valor.atoms(sp.Float) or any(abs(r.q) > 10**6 for r in valor.atoms(sp.Rational)):
            return sp.Float(z, 15), "exacta"     # extremos decimales largos (un cruce numérico): el valor en decimal
        return simplifica(valor), "exacta"
    return sp.Float(_quad(numerica(f), float(a), float(b)), 15), "numérica"


class Integrador:
    """Integra varias funciones en una misma llamada. En cuanto sympy falla o tarda una vez, las demás integrales
    van directo a quad; así una calculadora no espera el límite de tiempo en cada integral."""
    def __init__(self, segundos=8):
        self.segundos, self.solo_numerica, self.metodos = segundos, False, set()

    def __call__(self, f, a, b):
        v, m = integra(f, a, b, self.segundos, self.solo_numerica)
        self.metodos.add(m)
        self.solo_numerica = self.solo_numerica or m == "numérica"
        return v

    @property
    def metodo(self):
        return "numérica" if "numérica" in self.metodos else "exacta"


def cruces(h, a, b):
    """Puntos de (a, b) donde h cambia de signo: exactos con solveset; si no, en una malla y con brentq.
    Devuelve (lista, "exactos" o "numéricos")."""
    try:
        sol = con_limite(lambda: sp.solveset(h, x, sp.Interval.open(a, b)), 8)
    except (ValueError, NotImplementedError, TypeError):
        sol = None
    if sol == sp.EmptySet:
        return [], "exactos"
    if isinstance(sol, sp.FiniteSet) and len(sol) <= 50 and all(c.is_real for c in sol):
        return sorted(sol, key=float), "exactos"
    fn = numerica(h)
    xs = malla_interior(a, b, 4001)
    signo = np.sign(fn(xs))
    raices, previo = [], None                       # previo: índice del último punto con signo distinto de 0
    for k, s_ in enumerate(signo):
        if s_ == 0 or np.isnan(s_):
            continue
        if previo is not None and s_ != signo[previo]:
            if k == previo + 1:
                raices.append(brentq(lambda t: float(fn(t)), xs[previo], xs[k], xtol=1e-14))
            else:                                   # h = 0 justo en la malla
                raices.append(xs[(previo + k) // 2])
        previo = k
    return [sp.Float(r, 15) for r in raices], "numéricos"


COMO = {"exacta": "cálculo exacto con sympy", "numérica": "cálculo numérico con quad"}


def punto(v, n=CIFRAS):
    """Un extremo o un cruce legible: exacto si es corto (pi/4, 3/2); si no, redondeado a n cifras."""
    if isinstance(v, sp.Basic) and not isinstance(v, sp.Float) and len(str(v)) <= 12:
        return str(v)
    return cifras(v, n)


def rotula(ax, xv, yv, texto, **kw):
    """Etiqueta directa junto a un punto de una curva (legible en blanco y negro, sin leyenda)."""
    kw = {"xytext": (4, 0), "textcoords": "offset points", "va": "center", "fontsize": 9, **kw}
    ax.annotate(texto, (float(xv), float(yv)), **kw)

## 6.1 Área entre curvas

$$A=\int_a^b\big|f(x)-g(x)\big|\,dx=\sum_k\left|\int_{c_k}^{c_{k+1}}\big(f(x)-g(x)\big)\,dx\right|,$$

con $c_0=a$, $c_m=b$ y, entre ellos, los cruces de las curvas. Entre dos cruces $f-g$ no cambia de signo, así que cada parte se integra sin valor absoluto y se suma su magnitud. La integral con signo $\int_a^b(f-g)\,dx$ resta las partes donde $g$ queda arriba.

En el ejemplo exacto de abajo, $y=4x$ y $y=x^3$ se cruzan en $x=2$ dentro de $[0,3]$.

In [ ]:
h = 4 * x - x**3
print("cruces en (0, 3):", sp.solveset(h, x, sp.Interval.open(0, 3)))
I1, I2 = sp.integrate(h, (x, 0, 2)), sp.integrate(h, (x, 2, 3))
print("∫_0^2 (4x - x³) dx =", I1, "   ∫_2^3 (4x - x³) dx =", I2)
print("integral con signo =", I1 + I2, "   área =", abs(I1) + abs(I2))

**Calculadora de área entre curvas.** Escribe $f$, $g$ y los extremos ($a$ y $b$ aceptan `pi`, `pi/2` o fracciones). La calculadora busca los cruces dentro de $(a,b)$, integra $f-g$ en cada parte con su signo y suma las magnitudes. Si `sympy` no encuentra los cruces o la integral exacta, los calcula numéricamente y te lo dice. En la gráfica, el gris marca dónde $f$ queda arriba y el achurado dónde $g$ queda arriba.

In [ ]:
def area_entre(f, g, a, b):
    """Área entre y = f(x) y y = g(x) en [a, b]. Devuelve un dict con los cruces dentro de (a, b), las partes
    [(p, q, integral con signo de f - g)], el área (suma de magnitudes), la integral con signo total y el método
    de integración. Un punto donde f - g solo toca 0 sin cambiar de signo no cuenta como cruce."""
    f, g = sp.sympify(f), sp.sympify(g)
    a, b = exacto(a, "a"), exacto(b, "b")
    if not a < b:
        raise ValueError("a debe ser menor que b.")
    h = f - g
    revisa_intervalo(h, a, b, "f - g")
    if simplifica(h) == 0:
        raise ValueError("f y g son la misma función: no encierran ninguna región (el área vale 0).")
    puntos, como = cruces(h, a, b)
    integ = Integrador()
    integ.solo_numerica = len(puntos) > 20          # f - g oscila mucho: todas las partes con quad
    partes = []
    for p, q in zip([a, *puntos], [*puntos, b]):
        I = integ(h, p, q)
        if partes and (float(partes[-1][2]) >= 0) == (float(I) >= 0):   # h solo tocó 0: se une con la parte anterior
            p0, _, I0 = partes.pop()
            p, I = p0, simplifica(I0 + I)
        partes.append((p, q, I))
    puntos = [q for _, q, _ in partes[:-1]]
    return {"cruces": puntos, "partes": partes, "area": simplifica(sum(abs(I) for _, _, I in partes)),
            "integral": simplifica(sum(I for _, _, I in partes)), "metodo": integ.metodo,
            "cruces_numericos": como == "numéricos" and bool(puntos)}


def calculadora_area(f_txt, g_txt, a_txt, b_txt):
    try:
        f, g = parsear(f_txt), parsear(g_txt)
        a, b = exacto(a_txt, "a"), exacto(b_txt, "b")
        r = area_entre(f, g, a, b)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; si x y y están en metros, las áreas salen en m²)")
    print("cruces dentro de (a, b):", ", ".join(punto(c) for c in r["cruces"]) or "ninguno")
    if len(r["partes"]) > 12:
        print(f"   ({len(r['partes'])} partes; se muestran las primeras 12)")
    for p, q, I in r["partes"][:12]:
        arriba = "f arriba" if float(I) > 0 else "g arriba"
        print(f"   ∫ (f - g) dx de {punto(p)} a {punto(q)} = {muestra(I)} u²   ({arriba})")
    print(f"área total = {muestra(r['area'])} u²")
    print(f"integral con signo = {muestra(r['integral'])} u²   ({COMO[r['metodo']]}"
          + ("; cruces con brentq, numéricos" if r["cruces_numericos"] else "") + ")")
    try:
        xs = np.linspace(float(a), float(b), 600)
        yf, yg = para_graficar(f, xs), para_graficar(g, xs)
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        ax.fill_between(xs, yg, yf, where=yf >= yg, interpolate=True, color="0.8")
        ax.fill_between(xs, yg, yf, where=yf < yg, interpolate=True, facecolor="white", edgecolor="black", hatch="///")
        ax.plot(xs, yf, color="black", ls="-"); ax.plot(xs, yg, color="black", ls="--")
        rotula(ax, xs[-2], yf[-2], "f"); rotula(ax, xs[-2], yg[-2], "g")
        for c in r["cruces"][:50]:
            ax.axvline(float(c), color="0.4", ls=":", lw=0.8)
        ax.axhline(0, color="black", lw=0.5); ax.set_xlabel("x")
        ax.set_title("gris: f arriba de g · achurado: g arriba de f", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_area,
    f_txt=widgets.Text(value="4*x", description="f(x) =", continuous_update=False),
    g_txt=widgets.Text(value="x^3", description="g(x) =", continuous_update=False),
    a_txt=widgets.Text(value="0", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="3", description="b =", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 6.1 (resultado conocido)
def _area_por_cuadratura(f, g, a, b):
    fn = numerica(f - g)
    return quad(lambda t: abs(float(fn(t))), float(a), float(b), limit=200)[0]


PRUEBAS_1 = [
    ("x + 2 y x² en [-1, 2]: área 9/2, sin cruces dentro",
     lambda: (lambda r: r["area"] == sp.Rational(9, 2) and r["cruces"] == [])(area_entre(x + 2, x**2, -1, 2))),
    ("4x y x³ en [0, 3]: cruce en 2, área 41/4, integral con signo -9/4",
     lambda: (lambda r: r["cruces"] == [2] and r["area"] == sp.Rational(41, 4) and r["integral"] == sp.Rational(-9, 4))(
         area_entre(4 * x, x**3, 0, 3))),
    ("sen x y cos x en [0, π]: cruce en π/4, área 2√2",
     lambda: (lambda r: r["cruces"] == [sp.pi / 4] and sp.simplify(r["area"] - 2 * sp.sqrt(2)) == 0)(
         area_entre(sp.sin(x), sp.cos(x), 0, sp.pi))),
    ("e^x y 3x en [0, 2]: dos cruces hallados numéricamente; el área coincide con ∫|f - g| por cuadratura",
     lambda: (lambda r: len(r["cruces"]) == 2 and r["cruces_numericos"]
              and cerca(r["area"], _area_por_cuadratura(sp.exp(x), 3 * x, 0, 2), 1e-8))(
         area_entre(sp.exp(x), 3 * x, 0, 2))),
    ("a ≥ b se rechaza", lambda: _rechaza(lambda: area_entre(x, x**2, 2, 1))),
    ("1/x y 0 en [-1, 1] se rechaza (asíntota en 0)", lambda: _rechaza(lambda: area_entre(1 / x, 0, -1, 1))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.1).** Calcula a mano el área de la región encerrada entre $y=x^2$ y $y=2x$ y escribe el número.

In [ ]:
mi_valor = None     # escribe un número (también sirve "4/3")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = area_entre(2 * x, x**2, 0, 2)["area"]
        print("La calculadora da:", muestra(ref), "u²")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: las curvas se cortan en x = 0 y x = 2; integra 2x - x² entre esos valores")

## 6.2 Volúmenes de sólidos de revolución

$$V_{\text{arandelas}}=\pi\int_a^b\Big(f(x)^2-g(x)^2\Big)\,dx\quad(\text{giro alrededor del eje }x),\qquad V_{\text{cascarones}}=2\pi\int_a^b x\,\big(f(x)-g(x)\big)\,dx\quad(\text{giro alrededor del eje }y).$$

Con arandelas, el radio exterior es la curva más alejada del eje y el interior la más cercana; si $g=0$ quedan discos. Con cascarones, $x$ es el radio de cada cascarón y $f-g$ su altura, así que la región debe quedar a la derecha del eje $y$ ($a\ge0$).

En el ejemplo exacto, el cono de radio $r$ y altura $h$ sale de girar $y=\frac{r}{h}x$ en $[0,h]$ alrededor del eje $x$.

In [ ]:
r_c, h_c = sp.symbols("r h", positive=True)
cono = sp.pi * sp.integrate((r_c / h_c * x)**2, (x, 0, h_c))
print("V del cono =", sp.factor(cono), "   con r = 3, h = 4:", cono.subs({r_c: 3, h_c: 4}))

**Calculadora de volúmenes.** Escribe $f$ y $g$ en cualquier orden (para discos, $g=0$), los extremos y el método. Con arandelas, en cada $x$ el radio exterior es el mayor de $|f|$ y $|g|$ y el interior el menor; la calculadora rechaza una región que quede a los dos lados del eje de giro, porque al girar sus dos partes se enciman. La gráfica dibuja la región en gris y su reflejo respecto al eje de giro, que es la silueta del sólido.

In [ ]:
def _suma_de_magnitudes(h, a, b, integ):
    """Σ |∫ h| sobre las partes de [a, b] donde h no cambia de signo (∫ |h| dx sin integrar un valor absoluto)."""
    puntos, _ = cruces(h, a, b)
    integ.solo_numerica = integ.solo_numerica or len(puntos) > 20
    return simplifica(sum(abs(integ(h, p, q)) for p, q in zip([a, *puntos], [*puntos, b])))


def volumen(f, g, a, b, metodo="arandelas"):
    """Volumen del sólido que genera la región entre y = f(x) y y = g(x) en [a, b] (f y g en cualquier orden).
    metodo "arandelas": giro alrededor del eje x; en cada x el radio exterior es el mayor de |f| y |g| y el interior
    el menor (discos si g = 0); se rechaza una región que quede a los dos lados del eje x.
    metodo "cascarones": giro alrededor del eje y; radio x y altura |f - g|; exige a >= 0.
    Devuelve un dict con V, el integrando (sin el valor absoluto) y el método de integración."""
    if metodo not in ("arandelas", "cascarones"):
        raise ValueError('el método es "arandelas" o "cascarones".')
    f, g = sp.sympify(f), sp.sympify(g)
    a, b = exacto(a, "a"), exacto(b, "b")
    if not a < b:
        raise ValueError("a debe ser menor que b.")
    revisa_intervalo(f, a, b, "f"); revisa_intervalo(g, a, b, "g")
    if simplifica(f - g) == 0:
        raise ValueError("f y g son la misma función: no encierran ninguna región (el volumen vale 0).")
    integ = Integrador()
    if metodo == "arandelas":
        xs = malla_interior(a, b)
        F, G = numerica(f)(xs), numerica(g)(xs)
        tol = 1e-9 * max(1.0, float(np.max(np.abs(F))), float(np.max(np.abs(G))))
        lados = ((F > tol) & (G < -tol)) | ((F < -tol) & (G > tol))
        if np.any(lados):
            raise ValueError(f"en x ≈ {cifras(xs[int(np.argmax(lados))], 4)} la región queda a los dos lados del eje x "
                             "(f y g tienen signos opuestos): al girar, la parte de arriba y la de abajo se enciman. "
                             "Calcula cada parte por separado; el sólido lo genera la más alejada del eje.")
        integrando = sp.pi * (f**2 - g**2)
    else:
        if a < 0:
            raise ValueError("con cascarones el radio es x y no puede ser negativo: usa a ≥ 0, con la región a la "
                             "derecha del eje y.")
        integrando = 2 * sp.pi * x * (f - g)
    V = _suma_de_magnitudes(sp.expand(integrando), a, b, integ)
    return {"V": V, "integrando": integrando, "metodo": integ.metodo}


def calculadora_volumen(f_txt, g_txt, a_txt, b_txt, metodo):
    try:
        f, g = parsear(f_txt), parsear(g_txt)
        a, b = exacto(a_txt, "a"), exacto(b_txt, "b")
        r = volumen(f, g, a, b, metodo)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    eje = "eje x" if metodo == "arandelas" else "eje y"
    print(f"({CIFRAS} cifras significativas, redondeadas; si x y y están en metros, el volumen sale en m³)")
    print(f"giro alrededor del {eje}:  V = ∫ de {punto(a)} a {punto(b)} de  |{r['integrando']}|  dx")
    print(f"V = {muestra(r['V'])} u³   ({COMO[r['metodo']]})")
    try:
        xs = np.linspace(float(a), float(b), 400)
        yf, yg = para_graficar(f, xs), para_graficar(g, xs)
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        k1, k2 = len(xs) * 2 // 5, len(xs) * 7 // 10
        if metodo == "arandelas":
            for s_ in (1, -1):
                ax.fill_between(xs, s_ * yg, s_ * yf, color="0.8" if s_ == 1 else "0.92")
                ax.plot(xs, s_ * yf, color="black", ls="-"); ax.plot(xs, s_ * yg, color="black", ls="--")
            ax.axhline(0, color="black", lw=0.8, ls="-.")
            rotula(ax, xs[-1], 0, "eje de giro", ha="right", xytext=(0, 7))
        else:
            for s_ in (1, -1):
                ax.fill_between(s_ * xs, yg, yf, color="0.8" if s_ == 1 else "0.92")
                ax.plot(s_ * xs, yf, color="black", ls="-"); ax.plot(s_ * xs, yg, color="black", ls="--")
            ax.axvline(0, color="black", lw=0.8, ls="-.")
            rotula(ax, 0, float(np.nanmax(np.concatenate((yf, yg)))), "eje de giro", xytext=(4, 6))
        rotula(ax, xs[k1], yf[k1], "f", xytext=(0, 9), ha="center")
        rotula(ax, xs[k2], yg[k2], "g", xytext=(0, -9), ha="center")
        ax.set_xlabel("x"); ax.set_title(f"región (gris) y su reflejo: silueta del sólido, {metodo}", fontsize=9)
        plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_volumen,
    f_txt=widgets.Text(value="sqrt(x)", description="f(x) =", continuous_update=False),
    g_txt=widgets.Text(value="x/2", description="g(x) =", continuous_update=False),
    a_txt=widgets.Text(value="0", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="4", description="b =", continuous_update=False),
    metodo=widgets.Dropdown(options=[("arandelas o discos (eje x)", "arandelas"), ("cascarones (eje y)", "cascarones")],
                            value="arandelas", description="método"));

In [ ]:
# Casos de prueba de la sección 6.2 (resultado conocido)
PRUEBAS_2 = [
    ("discos de y = 3x/4 en [0, 4]: cono de 12π", lambda: volumen(3 * x / 4, 0, 0, 4, "arandelas")["V"] == 12 * sp.pi),
    ("arandelas con √x y x/2 en [0, 4], en cualquier orden: 8π/3",
     lambda: volumen(sp.sqrt(x), x / 2, 0, 4, "arandelas")["V"] == volumen(x / 2, sp.sqrt(x), 0, 4, "arandelas")["V"]
     == sp.Rational(8, 3) * sp.pi),
    ("cascarones con 4x - x² en [0, 4]: 128π/3",
     lambda: volumen(4 * x - x**2, 0, 0, 4, "cascarones")["V"] == sp.Rational(128, 3) * sp.pi),
    ("región bajo el eje x entre -x y -2x en [0, 1]: radios x y 2x, V = π",
     lambda: volumen(-x, -2 * x, 0, 1, "arandelas")["V"] == sp.pi),
    ("discos de y = x en [-1, 1]: dos conos, V = 2π/3", lambda: volumen(x, 0, -1, 1, "arandelas")["V"] == 2 * sp.pi / 3),
    ("se rechazan una región a los dos lados del eje x (1 y -1) y cascarones con a < 0",
     lambda: _rechaza(lambda: volumen(1, -1, 0, 1, "arandelas"))
     and _rechaza(lambda: volumen(1 - x**2, 0, -1, 1, "cascarones"))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.2).** Calcula a mano, por discos, el volumen de la esfera de radio 1 que se forma al girar $y=\sqrt{1-x^2}$ en $[-1,1]$ alrededor del eje $x$, y escribe el número.

In [ ]:
mi_valor = None     # escribe un número (también sirve "4*pi/3")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = volumen(sp.sqrt(1 - x**2), 0, -1, 1, "arandelas")["V"]
        print("La calculadora da:", muestra(ref), "u³")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: el radio del disco es √(1 - x²), así que su área es π(1 - x²); integra de -1 a 1")

## 6.3 Longitud de arco

$$L=\int_a^b\sqrt{1+\big(f'(x)\big)^2}\,dx,\qquad L_n=\sum_{k=1}^{n}\sqrt{(\Delta x)^2+(\Delta y_k)^2}\quad(\text{poligonal de }n\text{ segmentos iguales en }x).$$

Cada segmento de la poligonal es una cuerda, más corta que el arco que une sus extremos, así que $L_n\le L$ y $L_n\to L$ cuando $n$ crece. La integral casi nunca tiene antiderivada elemental; cuando no la tiene, se calcula numéricamente.

El ejemplo exacto usa $y=x^{3/2}$ en $[0,4]$, cuyo integrando $\sqrt{1+\tfrac94x}$ sí tiene antiderivada.

In [ ]:
y_ej = x**sp.Rational(3, 2)
integrando_ej = sp.sqrt(1 + sp.diff(y_ej, x)**2)
L_ej = sp.integrate(integrando_ej, (x, 0, 4))
print("√(1 + f'²) =", integrando_ej, "   L =", L_ej, "≈", cifras(L_ej, 6))

**Calculadora de longitud de arco.** Escribe $f$, los extremos y el número $n$ de segmentos. La calculadora intenta la integral exacta con `sympy` (con límite de tiempo); si no sale, la calcula con `quad` y te dice cuál usó. Mueve $n$ para ver cómo la poligonal se acerca a la curva por debajo.

In [ ]:
def longitud_arco(f, a, b):
    """L = ∫_a^b √(1 + f'²) dx. Devuelve un dict con L, el integrando y el método (exacto con sympy o quad)."""
    f = sp.sympify(f)
    a, b = exacto(a, "a"), exacto(b, "b")
    if not a < b:
        raise ValueError("a debe ser menor que b.")
    revisa_intervalo(f, a, b, "f")
    df = sp.diff(f, x)
    if f.has(sp.Heaviside) or df.has(sp.DiracDelta):
        raise ValueError("f tiene un salto (un escalón): la longitud de arco pide una curva continua con f' continua, "
                         "al menos por tramos. Mide cada tramo continuo por separado.")
    try:                                   # factor: a veces 1 + f'² es un cuadrado perfecto (catenaria)
        integrando = con_limite(lambda: sp.sqrt(sp.factor(sp.together(1 + df**2))), 5)
    except ValueError:
        integrando = sp.sqrt(1 + df**2)
    L, como = integra(integrando, a, b)
    return {"L": L, "integrando": integrando, "metodo": como}


def poligonal(f, a, b, n):
    """Longitud de la poligonal de n segmentos con vértices sobre y = f(x), igualmente espaciados en x."""
    if int(n) != n or n < 1:
        raise ValueError("n debe ser un entero mayor o igual que 1.")
    a, b = float(exacto(a, "a")), float(exacto(b, "b"))
    if not a < b:
        raise ValueError("a debe ser menor que b.")
    xs = np.linspace(a, b, int(n) + 1)
    ys = numerica(sp.sympify(f))(xs)
    if not np.all(np.isfinite(ys)):
        raise ValueError("f no tiene valor real en algún vértice de la poligonal: revisa el dominio.")
    return float(np.sum(np.hypot(np.diff(xs), np.diff(ys))))


def calculadora_arco(f_txt, a_txt, b_txt, n):
    try:
        f = parsear(f_txt)
        a, b = exacto(a_txt, "a"), exacto(b_txt, "b")
        r = longitud_arco(f, a, b)
        Ln = poligonal(f, a, b, n)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; si x y y están en metros, L sale en m)")
    print(f"integrando √(1 + f'²) = {r['integrando']}")
    print(f"{'L (integral)':<20} = {muestra(r['L'])} u   ({COMO[r['metodo']]})")
    print(f"{f'L_{n} (poligonal)':<20} = {cifras(Ln, CIFRAS)} u")
    print(f"{f'diferencia L - L_{n}':<20} = {cifras(float(r['L']) - Ln, 3)} u   (3 cifras)")
    try:
        xs = np.linspace(float(a), float(b), 600)
        xp = np.linspace(float(a), float(b), n + 1)
        ys, yp = para_graficar(f, xs), para_graficar(f, xp)
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        ax.plot(xs, ys, color="black", ls="-")
        ax.plot(xp, yp, color="0.35", ls="--", marker="o" if n <= 32 else None, ms=3)
        rotula(ax, xs[-1], ys[-1], "curva")
        k = max(0, n // 2 - 1)
        rotula(ax, (xp[k] + xp[k + 1]) / 2, (yp[k] + yp[k + 1]) / 2, f"poligonal, n = {n}", xytext=(6, -10), va="top")
        ax.set_xlabel("x"); ax.set_title("arco (continua) y poligonal inscrita (discontinua)", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_arco,
    f_txt=widgets.Text(value="x^(3/2)", description="f(x) =", continuous_update=False),
    a_txt=widgets.Text(value="0", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="4", description="b =", continuous_update=False),
    n=widgets.IntSlider(value=4, min=1, max=64, description="n", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 6.3 (resultado conocido)
PRUEBAS_3 = [
    ("y = x^(3/2) en [0, 4]: (80√10 - 8)/27 ≈ 9.07342, exacta",
     lambda: (lambda r: r["metodo"] == "exacta" and sp.simplify(r["L"] - (80 * sp.sqrt(10) - 8) / 27) == 0
              and cifras(r["L"], 6) == "9.07342")(longitud_arco(x**sp.Rational(3, 2), 0, 4))),
    ("cable parabólico y = 4·20·x²/200² en [-100, 100]: 205.212 m",
     lambda: round(float(longitud_arco(4 * 20 * x**2 / 200**2, -100, 100)["L"]), 3) == 205.212),
    ("catenaria y = (e^x + e^(-x))/2 en [-1, 1]: e - 1/e",
     lambda: cerca(longitud_arco((sp.exp(x) + sp.exp(-x)) / 2, -1, 1)["L"], math.e - 1 / math.e, 1e-12)),
    ("poligonal de y = x^(3/2) con n = 64: a menos de 1e-3 de la exacta, y por debajo",
     lambda: 0 <= float((80 * sp.sqrt(10) - 8) / 27) - poligonal(x**sp.Rational(3, 2), 0, 4, 64) < 1e-3),
    ("poligonal de una recta con n = 1: exacta (y = 3x + 1 en [0, 4] → 4√10)",
     lambda: cerca(poligonal(3 * x + 1, 0, 4, 1), 4 * math.sqrt(10), 1e-12)),
    ("y = sen x en [0, π]: sin antiderivada elemental, se usa quad (3.82020)",
     lambda: (lambda r: r["metodo"] == "numérica" and cifras(r["L"], 6) == "3.82020")(longitud_arco(sp.sin(x), 0, sp.pi))),
    ("n = 0 se rechaza", lambda: _rechaza(lambda: poligonal(x**2, 0, 1, 0))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.3).** Calcula a mano la longitud de la recta $y=2x$ entre $x=0$ y $x=3$ y escribe el número.

In [ ]:
mi_valor = None     # escribe un número (también sirve "3*sqrt(5)")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = longitud_arco(2 * x, 0, 3)["L"]
        print("La calculadora da:", muestra(ref), "u")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: con f' = 2 el integrando es √(1 + 4) = √5, constante; multiplica por la longitud del intervalo")

## 6.4 Centroides y momentos de inercia de área

Para la región entre $g$ (abajo) y $f$ (arriba) en $[a,b]$:

$$A=\int_a^b(f-g)\,dx,\qquad \bar x=\frac1A\int_a^b x\,(f-g)\,dx,\qquad \bar y=\frac1A\int_a^b\frac{f^2-g^2}{2}\,dx,$$

$$I_x=\int_a^b\frac{f^3-g^3}{3}\,dx\ \ (\text{respecto al eje }x),\qquad I_{\bar x}=I_x-A\,\bar y^{\,2}\ \ (\text{teorema de Steiner}).$$

Para una sección hecha de rectángulos $b_i\times h_i$ con centro a la altura $y_i$: $\bar y=\dfrac{\sum A_iy_i}{\sum A_i}$ e $I_{\bar x}=\sum\left(\dfrac{b_ih_i^3}{12}+A_i\,d_i^2\right)$, con $d_i=y_i-\bar y$. El término $A_id_i^2$ traslada la inercia de cada rectángulo al eje centroidal de la sección; sin él la viga parece mucho más flexible de lo que es.

El ejemplo exacto usa la región bajo $y=4-x^2$ en $[0,2]$.

In [ ]:
f_ej = 4 - x**2
A_ej = sp.integrate(f_ej, (x, 0, 2))
xb_ej = sp.integrate(x * f_ej, (x, 0, 2)) / A_ej
yb_ej = sp.integrate(f_ej**2 / 2, (x, 0, 2)) / A_ej
print("A =", A_ej, "  x̄ =", xb_ej, "  ȳ =", yb_ej)

**Calculadora de una región entre curvas.** Escribe $f$ (arriba), $g$ (abajo) y los extremos, con longitudes en cm. La calculadora da el área, el centroide, $I_x$ respecto al eje $x$ e $I_{\bar x}$ respecto al eje horizontal que pasa por el centroide, y los dibuja.

In [ ]:
def region(f, g, a, b):
    """Área, centroide (x̄, ȳ), I_x respecto al eje x e I_c respecto al eje centroidal horizontal (Steiner)
    de la región entre g (abajo) y f (arriba) en [a, b]. Devuelve un dict. Si una región no acotada tiene área
    finita pero x̄, ȳ o los momentos divergen (como bajo 1/√x en [0, 1]), esos valores son None y la clave
    "divergen" dice cuáles."""
    f, g = sp.sympify(f), sp.sympify(g)
    a, b = exacto(a, "a"), exacto(b, "b")
    if not a < b:
        raise ValueError("a debe ser menor que b.")
    revisa_intervalo(f, a, b, "f"); revisa_intervalo(g, a, b, "g")
    xs = malla_interior(a, b)
    dif = numerica(f)(xs) - numerica(g)(xs)
    if np.any(dif < -1e-9 * max(1.0, float(np.max(np.abs(dif))))):
        raise ValueError(f"en x ≈ {cifras(xs[int(np.argmin(dif))], 4)} la curva g queda arriba de f. Escribe arriba "
                         "la curva superior (f) o separa la región en el cruce.")
    integ = Integrador()
    A = integ(sp.expand(f - g), a, b)
    if float(A) <= 0:
        raise ValueError("la región tiene área cero: f y g coinciden en todo el intervalo.")
    def _opcional(h):
        try:
            return integ(sp.expand(h), a, b)
        except ValueError:                          # diverge: el resto del resultado sigue valiendo
            return None
    Mx, My, Ix = _opcional(x * (f - g)), _opcional((f**2 - g**2) / 2), _opcional((f**3 - g**3) / 3)
    xb = simplifica(Mx / A) if Mx is not None else None
    yb = simplifica(My / A) if My is not None else None
    Ic = simplifica(Ix - A * yb**2) if Ix is not None and yb is not None else None
    divergen = [n for n, v in (("x̄", xb), ("ȳ", yb), ("I_x", Ix), ("I_c", Ic)) if v is None]
    return {"A": A, "x_bar": xb, "y_bar": yb, "I_x": Ix, "I_c": Ic, "metodo": integ.metodo, "divergen": divergen}


def calculadora_region(f_txt, g_txt, a_txt, b_txt):
    try:
        f, g = parsear(f_txt), parsear(g_txt)
        a, b = exacto(a_txt, "a"), exacto(b_txt, "b")
        r = region(f, g, a, b)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    def _v(clave, unidad):
        return f"{muestra(r[clave])} {unidad}" if r[clave] is not None else "no existe: su integral diverge"
    print(f"({CIFRAS} cifras significativas, redondeadas; longitudes en cm; {COMO[r['metodo']]})")
    print(f"A   = {muestra(r['A'])} cm²")
    print(f"x̄   = {_v('x_bar', 'cm')}     ȳ = {_v('y_bar', 'cm')}")
    print(f"I_x = {_v('I_x', 'cm⁴')}   (respecto al eje x)")
    print(f"I_c = {_v('I_c', 'cm⁴')}   (respecto al eje centroidal horizontal, I_x - A ȳ²)")
    if r["divergen"]:
        print("La región no está acotada cerca de un extremo: su área es finita, pero", ", ".join(r["divergen"]),
              "no, porque usan f² o f³, que crecen más rápido cerca de la asíntota.")
    try:
        xs = np.linspace(float(a), float(b), 400)
        yf, yg = para_graficar(f, xs), para_graficar(g, xs)
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        ax.fill_between(xs, yg, yf, color="0.85")
        ax.plot(xs, yf, color="black", ls="-"); ax.plot(xs, yg, color="black", ls="--")
        k1, k2 = len(xs) // 4, len(xs) * 3 // 4
        rotula(ax, xs[k1], yf[k1], "f", xytext=(0, 9), ha="center")
        rotula(ax, xs[k2], yg[k2], "g", xytext=(0, -9), ha="center")
        if r["x_bar"] is not None and r["y_bar"] is not None:
            xb, yb = float(r["x_bar"]), float(r["y_bar"])
            ax.axhline(yb, color="0.3", ls="-.", lw=0.8); rotula(ax, xs[-1], yb, "eje centroidal", xytext=(6, 6))
            ax.plot([xb], [yb], marker="+", color="black", ms=14, mew=2)
            rotula(ax, xb, yb, f"C ({cifras(xb, 3)}, {cifras(yb, 3)})", xytext=(8, -10))
        ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)"); ax.set_aspect("equal", adjustable="datalim")
        ax.set_title("C: centroide (coordenadas con 3 cifras)", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_region,
    f_txt=widgets.Text(value="4 - x^2", description="f(x) =", continuous_update=False),
    g_txt=widgets.Text(value="0", description="g(x) =", continuous_update=False),
    a_txt=widgets.Text(value="0", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="2", description="b =", continuous_update=False));

**Calculadora de una sección compuesta de rectángulos.** Escribe cada rectángulo como `b, h, y_centro` en cm (base, altura y altura de su centro); deja vacío el que no uses. Los valores de ejemplo forman una sección T: patín de $12\times2$ con centro en $y=11$ y alma de $2\times10$ con centro en $y=5$. La calculadora suma los rectángulos con el teorema de Steiner y también muestra la suma sin el término $A_id_i^2$, para que veas el tamaño del error.

In [ ]:
def seccion_compuesta(rectangulos):
    """Sección hecha de rectángulos (b, h, y_centro): área total A, altura del centroide ȳ, momento de inercia I
    respecto al eje centroidal horizontal (Σ b h³/12 + A d²), la suma sin Steiner y el detalle por rectángulo."""
    rects = list(rectangulos)
    if not rects:
        raise ValueError("escribe al menos un rectángulo.")
    datos = []
    for k, rect in enumerate(rects, 1):
        if len(rect) != 3:
            raise ValueError(f"el rectángulo {k} necesita tres números: b, h, y_centro.")
        b, h, yc = (exacto(v, f"el dato del rectángulo {k}") for v in rect)
        if b <= 0 or h <= 0:
            raise ValueError(f"en el rectángulo {k} la base y la altura deben ser positivas.")
        datos.append((b, h, yc))
    A = sum(b * h for b, h, _ in datos)
    yb = sum(b * h * yc for b, h, yc in datos) / A
    detalle = [(b * h, yc - yb, b * h**3 / 12, b * h * (yc - yb)**2) for b, h, yc in datos]
    I_propias = sum(d[2] for d in detalle)
    I = I_propias + sum(d[3] for d in detalle)
    return {"A": A, "y_bar": yb, "I": I, "I_sin_steiner": I_propias, "detalle": detalle}


def _enciman(rects):
    """True si dos rectángulos ocupan la misma franja de alturas (centrados en el mismo eje, se enciman)."""
    franjas = sorted((yc - h / 2, yc + h / 2) for _, h, yc in rects)
    return any(s2 < e1 - 1e-12 for (_, e1), (s2, _) in zip(franjas, franjas[1:]))


def calculadora_seccion(r1_txt, r2_txt, r3_txt):
    try:
        rects = [lista_numeros(t, f"el rectángulo {k}") for k, t in enumerate((r1_txt, r2_txt, r3_txt), 1) if t.strip()]
        r = seccion_compuesta(rects)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; longitudes en cm)")
    print(" rect.   A_i (cm²)    d_i (cm)   b h³/12 (cm⁴)   A_i d_i² (cm⁴)")
    for k, (Ai, di, Ii, Ad2) in enumerate(r["detalle"], 1):
        print(f"  {k}   {cifras(Ai, CIFRAS):>10}  {cifras(di, CIFRAS):>10}  {cifras(Ii, CIFRAS):>14}  {cifras(Ad2, CIFRAS):>14}")
    print(f"A = {cifras(r['A'], CIFRAS)} cm²     ȳ = {cifras(r['y_bar'], CIFRAS)} cm (desde y = 0)")
    print(f"I respecto al eje centroidal = {cifras(r['I'], CIFRAS)} cm⁴")
    if r["I"] - r["I_sin_steiner"] > 0:
        falta = 100 * (1 - r["I_sin_steiner"] / r["I"])
        print(f"sin el término A d² saldría {cifras(r['I_sin_steiner'], CIFRAS)} cm⁴, un {cifras(falta, 3)} % menos "
              "(porcentaje con 3 cifras): ese resultado es incorrecto")
    if _enciman(rects):
        print("Aviso: dos rectángulos ocupan la misma franja de alturas. Si en la pieza se tocan, el área común "
              "se está contando dos veces.")
    fig, ax = plt.subplots(figsize=(4.5, 3.6))
    for (b, h, yc), achurado in zip(rects, ("///", "\\\\\\", "xxx")):
        ax.add_patch(plt.Rectangle((-b / 2, yc - h / 2), b, h, facecolor="white", edgecolor="black", hatch=achurado))
    yb = float(r["y_bar"])
    ancho = max(b for b, _, _ in rects)
    ax.axhline(yb, color="black", ls="--", lw=1)
    rotula(ax, ancho / 2, yb, f"eje centroidal, ȳ = {cifras(yb, CIFRAS)} cm", xytext=(6, 8))
    ax.autoscale(); ax.set_aspect("equal"); ax.margins(0.15)
    ax.set_xlabel("cm"); ax.set_ylabel("y (cm)"); plt.show()


_ancho = {"description_width": "initial"}
widgets.interact(calculadora_seccion,
    r1_txt=widgets.Text(value="12, 2, 11", description="rect. 1 (b, h, y):", continuous_update=False, style=_ancho),
    r2_txt=widgets.Text(value="2, 10, 5", description="rect. 2 (b, h, y):", continuous_update=False, style=_ancho),
    r3_txt=widgets.Text(value="", description="rect. 3 (b, h, y):", continuous_update=False, style=_ancho));

In [ ]:
# Casos de prueba de la sección 6.4 (resultado conocido)
def _seccion(rects):
    return {k: float(v) for k, v in seccion_compuesta(rects).items() if k != "detalle"}


PRUEBAS_4 = [
    ("región bajo 4 - x² en [0, 2]: A = 16/3, x̄ = 3/4, ȳ = 8/5",
     lambda: (lambda r: (r["A"], r["x_bar"], r["y_bar"]) == (sp.Rational(16, 3), sp.Rational(3, 4), sp.Rational(8, 5)))(
         region(4 - x**2, 0, 0, 2))),
    ("rectángulo 4 × 6 (f = 6, g = 0 en [0, 4]): I_x = 288 e I centroidal = 72 = b h³/12",
     lambda: (lambda r: r["I_x"] == 288 and r["I_c"] == 72)(region(6 + 0 * x, 0, 0, 4))),
    ("triángulo de base 4 y altura 6 (bajo y = 3x/2 en [0, 4]): centroide (8/3, 2)",
     lambda: (lambda r: (r["x_bar"], r["y_bar"]) == (sp.Rational(8, 3), 2))(region(3 * x / 2, 0, 0, 4))),
    ("región con g arriba de f se rechaza", lambda: _rechaza(lambda: region(0, 4 - x**2, 0, 2))),
    ("sección T (patín 12 × 2 en y = 11, alma 2 × 10 en y = 5): ȳ = 8.2727, I = 567.39 cm⁴, sin Steiner 174.67",
     lambda: (lambda r: round(r["y_bar"], 4) == 8.2727 and round(r["I"], 2) == 567.39
              and round(r["I_sin_steiner"], 2) == 174.67)(_seccion([(12, 2, 11), (2, 10, 5)]))),
    ("sección I (patines 19 × 1 en y = ±14.5, alma 0.6 × 28): I = 9090.27 cm⁴, A = 54.8 cm²",
     lambda: (lambda r: round(r["I"], 2) == 9090.27 and cerca(r["A"], 54.8))(
         _seccion([(19, 1, 14.5), (19, 1, -14.5), (0.6, 28, 0)]))),
    ("rectángulo 4 × 30: I = 9000 cm⁴", lambda: seccion_compuesta([(4, 30, 15)])["I"] == 9000),
    ("las dos calculadoras coinciden: rectángulo 4 × 6 da 72 en ambas",
     lambda: seccion_compuesta([(4, 6, 3)])["I"] == region(6 + 0 * x, 0, 0, 4)["I_c"] == 72),
    ("base negativa se rechaza", lambda: _rechaza(lambda: seccion_compuesta([(-2, 10, 5)]))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.4).** Un triángulo rectángulo tiene base 6 sobre el eje $x$ y altura 3. Calcula a mano la altura $\bar y$ de su centroide y escribe el número.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = region(x / 2, 0, 0, 6)["y_bar"]          # hipotenusa y = x/2 de (0, 0) a (6, 3)
        print("La calculadora da: ȳ =", muestra(ref))
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: con f = x/2 y g = 0, ȳ = (1/A) ∫ f²/2 dx; el centroide de un triángulo queda a h/3 de la base")

## 6.5 Momentos de masa

Para una barra sobre $[0,L]$ con densidad lineal $\lambda(x)$ en kg/m:

$$m=\int_0^L\lambda(x)\,dx,\qquad \bar x=\frac1m\int_0^Lx\,\lambda(x)\,dx,\qquad I_O=\int_0^Lx^2\,\lambda(x)\,dx,\qquad I_G=I_O-m\,\bar x^{\,2}.$$

$I_O$ es el momento de inercia respecto a un eje perpendicular a la barra que pasa por el extremo $x=0$, e $I_G$ el que pasa por el centro de masa. Una masa puntual $m_p$ en la punta $x=L$ suma $m_p$ a la masa, $m_pL$ al primer momento y $m_pL^2$ a $I_O$.

El ejemplo exacto es una barra de $L=\tfrac12$ m con $\lambda(x)=4-4x$ kg/m, que se adelgaza hacia la punta.

In [ ]:
lam_ej, L_ej5 = 4 - 4 * x, sp.Rational(1, 2)
m_ej = sp.integrate(lam_ej, (x, 0, L_ej5))
print("m =", m_ej, "kg   x̄ =", sp.integrate(x * lam_ej, (x, 0, L_ej5)) / m_ej, "m   I_O =",
      sp.integrate(x**2 * lam_ej, (x, 0, L_ej5)), "kg·m²")

**Calculadora de una barra.** Escribe $\lambda(x)$ en kg/m, la longitud $L$ en m y, si la hay, la masa puntual en la punta en kg. La calculadora da la masa, el centro de masa y los momentos de inercia respecto al extremo y al centro de masa. En la gráfica, el área bajo $\lambda$ es la masa de la barra.

In [ ]:
def barra(lam, L, m_punta=0):
    """Barra sobre [0, L] con densidad lineal lam(x) en kg/m y una masa puntual opcional m_punta en x = L.
    Devuelve un dict con m (total), x_bar, I_O (respecto a x = 0), I_G (respecto al centro de masa) y m_barra."""
    lam = sp.sympify(lam)
    L, mp = exacto(L, "L"), exacto(m_punta, "la masa en la punta")
    if L <= 0:
        raise ValueError("la longitud L debe ser positiva.")
    if mp < 0:
        raise ValueError("la masa en la punta no puede ser negativa (escribe 0 si no hay).")
    revisa_intervalo(lam, 0, L, "λ")
    if lam.has(x):
        valores = numerica(lam)(malla_interior(0, L))
        if np.any(valores < -1e-12):
            raise ValueError(f"λ es negativa cerca de x ≈ {cifras(malla_interior(0, L)[int(np.argmin(valores))], 4)} m: "
                             "una densidad no puede ser negativa. Revisa la fórmula o la longitud.")
    elif lam < 0:
        raise ValueError("una densidad no puede ser negativa.")
    integ = Integrador()
    def _int(h):
        return integ(sp.expand(h), 0, L)
    m_b = _int(lam)
    m = m_b + mp
    if float(m) <= 0:
        raise ValueError("la masa total es cero: escribe una densidad positiva o una masa en la punta.")
    xb = simplifica((_int(x * lam) + mp * L) / m)
    IO = simplifica(_int(x**2 * lam) + mp * L**2)
    IG = simplifica(IO - m * xb**2)
    return {"m": simplifica(m), "x_bar": xb, "I_O": IO, "I_G": IG, "m_barra": m_b,
            "metodo": integ.metodo}


def calculadora_barra(lam_txt, L, m_punta):
    try:
        lam = parsear(lam_txt)
        r = barra(lam, L, m_punta)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; {COMO[r['metodo']]})")
    print(f"masa de la barra = {muestra(r['m_barra'])} kg" + (f"   masa total = {muestra(r['m'])} kg" if m_punta else ""))
    print(f"x̄   = {muestra(r['x_bar'])} m   (centro de masa, medido desde x = 0)")
    print(f"I_O = {muestra(r['I_O'])} kg·m²   (eje en x = 0)")
    print(f"I_G = {muestra(r['I_G'])} kg·m²   (eje en el centro de masa, I_O - m x̄²)")
    try:
        xs = np.linspace(0, float(L), 300)
        ys = para_graficar(lam, xs)
        fig, ax = plt.subplots(figsize=(5.5, 3))
        ax.fill_between(xs, 0, ys, color="0.85"); ax.plot(xs, ys, color="black")
        rotula(ax, xs[len(xs) // 3], ys[len(ys) // 3], "λ(x)", xytext=(4, 8))
        xb = float(r["x_bar"])
        ax.axvline(xb, color="black", ls="--", lw=1)
        rotula(ax, xb, float(np.nanmax(ys)) * 0.5, f"x̄ = {cifras(xb, CIFRAS)} m")
        if m_punta:
            ax.plot([float(L)], [0], marker="s", color="black", ms=9, clip_on=False)
            rotula(ax, float(L), 0, f"{m_punta:g} kg", xytext=(-6, 12), ha="right")
        ax.set_xlabel("x (m)"); ax.set_ylabel("λ (kg/m)"); ax.set_ylim(bottom=0)
        ax.set_title("el área gris es la masa de la barra", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_barra,
    lam_txt=widgets.Text(value="4 - 4x", description="λ(x) =", continuous_update=False),
    L=widgets.FloatText(value=0.5, description="L (m)"),
    m_punta=widgets.FloatText(value=0.5, description="punta (kg)"));

In [ ]:
# Casos de prueba de la sección 6.5 (resultado conocido)
PRUEBAS_5 = [
    ("λ = 2, L = 3: m = 6, x̄ = 1.5, I_O = 18",
     lambda: (lambda r: (r["m"], r["x_bar"], r["I_O"]) == (6, sp.Rational(3, 2), 18))(barra(2, 3))),
    ("λ = 4 - 4x, L = 1/2: m = 3/2, x̄ = 2/9, I_O = 5/48",
     lambda: (lambda r: (r["m"], r["x_bar"], r["I_O"]) == (sp.Rational(3, 2), sp.Rational(2, 9), sp.Rational(5, 48)))(
         barra(4 - 4 * x, sp.Rational(1, 2)))),
    ("λ = x, L = 1: m = 1/2, x̄ = 2/3, I_O = 1/4",
     lambda: (lambda r: (r["m"], r["x_bar"], r["I_O"]) == (sp.Rational(1, 2), sp.Rational(2, 3), sp.Rational(1, 4)))(
         barra(x, 1))),
    ("λ = 4 - 4x, L = 1/2 con 0.5 kg en la punta: I_O = 0.229167",
     lambda: cifras(barra(4 - 4 * x, 0.5, 0.5)["I_O"], 6) == "0.229167"),
    ("barra uniforme de 6 kg y 3 m: I_G = m L²/12 = 4.5", lambda: barra(2, 3)["I_G"] == sp.Rational(9, 2)),
    ("densidad negativa se rechaza", lambda: _rechaza(lambda: barra(1 - 2 * x, 1))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.5).** Una barra uniforme de 2 kg mide 1 m. Calcula a mano su momento de inercia $I_O$ respecto a un eje perpendicular que pasa por un extremo y escribe el número en kg·m².

In [ ]:
mi_valor = None     # escribe un número (también sirve "2/3")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = barra(2, 1)["I_O"]          # λ = 2 kg/m en [0, 1]
        print("La calculadora da: I_O =", muestra(ref), "kg·m²")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: con λ = m/L = 2 kg/m, I_O = ∫_0^1 2x² dx; recuerda que el resultado general es m L²/3")

## 6.6 De la aceleración a la posición

$$v(t)=v_0+\int_0^ta(\tau)\,d\tau,\qquad s(t)=s_0+\int_0^tv(\tau)\,d\tau.$$

Con datos medidos cada $\Delta t$, las integrales se acumulan con la regla del trapecio: $v_{k+1}=v_k+\tfrac{\Delta t}{2}(a_k+a_{k+1})$ y lo mismo para $s$ a partir de $v$. Un acelerómetro con sesgo constante $b$ mide $a+b$; al integrar dos veces, el error en la posición crece como $e(t)=\tfrac12bt^2$, aunque $b$ sea pequeño. Por eso la navegación con acelerómetros necesita una referencia externa que la corrija.

El ejemplo exacto parte del reposo en el origen con $a(t)=6t$ m/s².

In [ ]:
t_s, tau_s = sp.symbols("t tau", nonnegative=True)
v_ej = sp.integrate(6 * tau_s, (tau_s, 0, t_s))
s_ej = sp.integrate(v_ej.subs(t_s, tau_s), (tau_s, 0, t_s))
print("v(t) =", v_ej, "m/s   s(t) =", s_ej, "m   s(2) =", s_ej.subs(t_s, 2), "m")

**Calculadora de movimiento.** Escribe $a(t)$ en m/s² (con `escalon(t - 2)` armas perfiles por tramos), la velocidad y la posición iniciales y el tiempo final. La calculadora integra dos veces con `sympy` y, aparte, simula un acelerómetro con sesgo $b$: integra con trapecios las muestras de $a+b$ y compara la posición medida con la verdadera. El ejemplo es el perfil de un vehículo que acelera 2 s, avanza 16 s a velocidad constante y frena 2 s.

In [ ]:
_TAU = sp.Dummy("tau", real=True)


class SinFormaExacta(ValueError):
    """sympy no encontró v(t) o s(t) como fórmula; la integral puede existir y calcularse con muestras."""


def movimiento(a, v0=0, s0=0):
    """v(t) y s(t) exactas (expresiones de sympy en x, que hace de t) a partir de a(t), v(0) = v0 y s(0) = s0.
    ValueError si la integral desde t = 0 diverge; SinFormaExacta si sympy no encuentra la fórmula (en ese caso
    integra muestras de a(t) con integra_datos)."""
    a = sp.sympify(a)
    v0, s0 = exacto(v0, "v0"), exacto(s0, "s0")
    if a.has(sp.Heaviside):
        a = a.rewrite(sp.Piecewise)
    def _acumula(g, cual):
        try:
            r = con_limite(lambda: sp.integrate(g.subs(x, _TAU), (_TAU, 0, x)), 10)
        except (ValueError, NotImplementedError, TypeError, ArithmeticError):
            raise SinFormaExacta("sympy no encontró una fórmula exacta para v(t) y s(t); integra muestras de a(t) "
                                 "con integra_datos.") from None
        if r.has(sp.oo, -sp.oo, sp.zoo, sp.nan):
            raise ValueError(f"la integral de {cual} desde t = 0 diverge: {cual} crece sin límite cerca de t = 0, así "
                             "que el movimiento no está definido con esos datos.")
        if r.has(sp.Integral, sp.exp_polar, sp.hyper, sp.meijerg):
            raise SinFormaExacta("sympy no encontró una fórmula exacta para v(t) y s(t); integra muestras de a(t) "
                                 "con integra_datos.")
        return r if r.has(sp.Piecewise, sp.Min, sp.Max) else sp.expand(r)
    v = v0 + _acumula(a, "a(t)")
    s = s0 + _acumula(v, "v(t)")
    return v, s


def integra_datos(t, a, v0=0.0, s0=0.0):
    """Velocidad y posición acumuladas con la regla del trapecio a partir de muestras a(t_k) (arreglos de numpy)."""
    t, a = np.asarray(t, dtype=float), np.asarray(a, dtype=float)
    if t.ndim != 1 or t.shape != a.shape or len(t) < 2:
        raise ValueError("t y a deben ser listas del mismo tamaño, con al menos dos muestras.")
    if not (np.all(np.isfinite(t)) and np.all(np.isfinite(a))):
        raise ValueError("hay muestras que no son números finitos.")
    dt = np.diff(t)
    if np.any(dt <= 0):
        raise ValueError("los tiempos deben ir en aumento.")
    v = v0 + np.concatenate(([0.0], np.cumsum(dt * (a[1:] + a[:-1]) / 2)))
    s = s0 + np.concatenate(([0.0], np.cumsum(dt * (v[1:] + v[:-1]) / 2)))
    return v, s


def error_sesgo(b, t):
    """Error en la posición, en m, por un sesgo constante b (m/s²) del acelerómetro después de t segundos."""
    return b * t**2 / 2


def tiempo_para_error(b, e_max):
    """Tiempo (s) en el que el error ½ b t² llega a e_max (m)."""
    if b <= 0 or e_max <= 0:
        raise ValueError("el sesgo y el error máximo deben ser positivos.")
    return math.sqrt(2 * e_max / b)


def _por_tramos(e):
    return e.has(sp.Piecewise, sp.Min, sp.Max, sp.Heaviside)


def calculadora_movimiento(a_txt, v0, s0, t_fin, b):
    try:
        a = parsear(a_txt)
        if not t_fin > 0:
            raise ValueError("el tiempo final debe ser positivo.")
        tf = exacto(t_fin, "el tiempo final")
        revisa_intervalo(a, 0, tf, "a(t)", var="t")
        try:
            v, s = movimiento(a, v0, s0)
        except SinFormaExacta:
            v = s = None
        ts = np.linspace(0, t_fin, 2001)
        a_k = numerica(a)(ts)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    t_sim = sp.Symbol("t")
    print(f"({CIFRAS} cifras significativas, redondeadas; t en s, a en m/s², v en m/s, s en m)")
    if v is None:
        print("sympy no encontró v(t) y s(t) como fórmula; los valores salen de integrar 2001 muestras con trapecios")
    elif _por_tramos(v) or _por_tramos(s):
        print("v(t) y s(t) quedan por tramos; la gráfica muestra s(t)")
    else:
        print(f"v(t) = {v.subs(x, t_sim)}     s(t) = {s.subs(x, t_sim)}")
    if v is not None:
        print(f"en t = {punto(tf)} s:  v = {muestra(simplifica(v.subs(x, tf)))} m/s   "
              f"s = {muestra(simplifica(s.subs(x, tf)))} m")
    if not np.all(np.isfinite(a_k)):
        print("a(t) no es finita en t = 0: se omite la simulación del acelerómetro, que usa muestras desde t = 0.")
        return
    v_ver, s_ver = integra_datos(ts, a_k, v0, s0)
    _, s_med = integra_datos(ts, a_k + b, v0, s0)
    if v is None:
        print(f"en t = {punto(tf)} s:  v ≈ {cifras(v_ver[-1], CIFRAS)} m/s   s ≈ {cifras(s_ver[-1], CIFRAS)} m")
    print(f"acelerómetro con sesgo b = {b:g} m/s²: posición medida {cifras(s_med[-1], CIFRAS)} m, "
          f"error {cifras(s_med[-1] - s_ver[-1], CIFRAS)} m  (½ b t² = {cifras(error_sesgo(b, t_fin), CIFRAS)} m)")
    if b > 0:
        print(f"el error llega a 0.5 m a los {cifras(tiempo_para_error(b, 0.5), 4)} s  (4 cifras)")
    try:
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        ax.plot(ts, s_ver, color="black", ls="-"); ax.plot(ts, s_med, color="black", ls="--")
        rotula(ax, ts[-1], s_ver[-1], "verdadera", va="top", xytext=(4, -4))
        rotula(ax, ts[-1], s_med[-1], "medida (con sesgo)", va="bottom", xytext=(4, 4))
        ax.set_xlabel("t (s)"); ax.set_ylabel("s (m)")
        ax.set_title("posición verdadera (continua) y medida con el sesgo (discontinua)", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_movimiento,
    a_txt=widgets.Text(value="0.5 - 0.5*escalon(t - 2) - 0.5*escalon(t - 18) + 0.5*escalon(t - 20)",
                       description="a(t) =", continuous_update=False, layout=widgets.Layout(width="95%")),
    v0=widgets.FloatText(value=0, description="v0 (m/s)"),
    s0=widgets.FloatText(value=0, description="s0 (m)"),
    t_fin=widgets.FloatText(value=20, description="t final (s)"),
    b=widgets.FloatSlider(value=0.05, min=0, max=0.1, step=0.005, readout_format=".3f",
                          description="sesgo b", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 6.6 (resultado conocido)
_perfil = sp.Piecewise((sp.Rational(1, 2), x < 2), (0, x < 18), (-sp.Rational(1, 2), x < 20), (0, True))
_ts60 = np.linspace(0, 60, 601)

PRUEBAS_6 = [
    ("a = 2 desde el reposo: s(3) = 9 m", lambda: movimiento(2, 0, 0)[1].subs(x, 3) == 9),
    ("a = 6t desde el reposo: v = 3t², s(2) = 8 m",
     lambda: (lambda vs: sp.expand(vs[0] - 3 * x**2) == 0 and vs[1].subs(x, 2) == 8)(movimiento(6 * x, 0, 0))),
    ("perfil por tramos (Piecewise): s(20) = 18 m y v(20) = 0",
     lambda: (lambda vs: vs[1].subs(x, 20) == 18 and vs[0].subs(x, 20) == 0)(movimiento(_perfil, 0, 0))),
    ("el mismo perfil escrito con escalones: s(20) = 18 m",
     lambda: movimiento(parsear("0.5 - 0.5*escalon(t - 2) - 0.5*escalon(t - 18) + 0.5*escalon(t - 20)"))[1].subs(x, 20) == 18),
    ("trapecios con a = 2 en [0, 3]: s(3) = 9 m (exacto para a constante)",
     lambda: cerca(integra_datos(np.linspace(0, 3, 31), np.full(31, 2.0))[1][-1], 9, 1e-9)),
    ("sesgo 0.05 m/s² durante 60 s: error de 90 m (fórmula y trapecios)",
     lambda: cerca(error_sesgo(0.05, 60), 90) and cerca(integra_datos(_ts60, np.full(601, 0.05))[1][-1], 90, 1e-9)),
    ("con b = 0.05 el error llega a 0.5 m en √20 ≈ 4.472 s", lambda: cifras(tiempo_para_error(0.05, 0.5), 4) == "4.472"),
    ("tiempos que no aumentan se rechazan", lambda: _rechaza(lambda: integra_datos([0, 2, 1], [1, 1, 1]))),
]
for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.6).** Una piedra cae desde el reposo con $a=g=9.81$ m/s² (hacia abajo positivo). Calcula a mano cuánto cae en 2 s y escribe el número en metros.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = movimiento(sp.Rational("9.81"), 0, 0)[1].subs(x, 2)
        print("La calculadora da: s(2) =", muestra(ref), "m")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: integra dos veces, v = g t y s = g t²/2; evalúa en t = 2 s")

## 6.7 Trabajo, energía y eficiencia

$$W=\int_{x_0}^{x_1}F(x)\,dx\quad(\text{J}=\text{N·m}),\qquad \eta=\frac{W_{\text{útil}}}{E_{\text{entrada}}},\quad 0<\eta\le1,\qquad 1\ \text{kWh}=3.6\times10^{6}\ \text{J}.$$

Para un resorte, $F=kx$ y $W=\tfrac12k\,(x_1^2-x_0^2)$. Al bombear un líquido, cada rebanada sube una distancia distinta: en un tanque cilíndrico de radio $r$ lleno hasta $H$, la rebanada a la altura $y$ pesa $\rho g\,\pi r^2\,dy$ y sube $H-y$, así que el integrando es $\rho g\,\pi r^2(H-y)$. Si una cuenta da $\eta>1$, hay un error en los datos o en las unidades, porque ninguna máquina entrega más energía de la que recibe.

El ejemplo exacto calcula el trabajo para estirar un resorte de $x_0$ a $x_1$.

In [ ]:
k_s, x0_s, x1_s = sp.symbols("k x_0 x_1", positive=True)
W_resorte = sp.integrate(k_s * x, (x, x0_s, x1_s))
print("W =", W_resorte, "   con k = 800 N/m, de 0.05 a 0.15 m:",
      W_resorte.subs({k_s: 800, x0_s: sp.Rational(1, 20), x1_s: sp.Rational(3, 20)}), "J")

**Calculadora de trabajo.** Escribe $F(x)$ en N (también sirve la variable $y$), los extremos en m y, si la conoces, la energía de entrada en J o kWh (0 si no hay dato). La calculadora integra la fuerza, da el trabajo en J y, con la energía de entrada, la eficiencia. El ejemplo es un malacate que sube 30 m una carga de 400 kg con un cable de 1.5 kg/m; a la altura $y$ quedan colgando $30-y$ metros de cable.

In [ ]:
def trabajo(F, x0, x1, con_metodo=False):
    """W = ∫_{x0}^{x1} F dx en J (F en N, x en m), exacto con sympy si se puede. Con con_metodo=True devuelve
    (W, método)."""
    F = sp.sympify(F)
    x0, x1 = exacto(x0, "x0"), exacto(x1, "x1")
    if x0 == x1:
        raise ValueError("x0 y x1 son iguales: no hay desplazamiento, el trabajo es 0.")
    revisa_intervalo(F, x0, x1, "F")
    W, como = integra(F, x0, x1)
    return (W, como) if con_metodo else W


def a_joules(valor, unidad="J"):
    """Energía en J a partir de J o kWh."""
    factores = {"J": 1, "kWh": 3600000}
    if unidad not in factores:
        raise ValueError('la unidad es "J" o "kWh".')
    return valor * factores[unidad]


def eficiencia(W_util, W_entrada):
    """η = W_util / W_entrada. ValueError si no cumple 0 < η <= 1."""
    W_util, W_entrada = float(W_util), float(W_entrada)
    if W_entrada <= 0:
        raise ValueError("la energía de entrada debe ser positiva.")
    if W_util <= 0:
        raise ValueError("el trabajo útil debe ser positivo para hablar de eficiencia.")
    eta = W_util / W_entrada
    if eta > 1:
        raise ValueError(f"saldría η = {cifras(eta, 4)} > 1: la máquina entregaría más energía de la que recibe, lo que "
                         "viola la conservación de la energía. Revisa los datos y las unidades (1 kWh = 3.6 MJ).")
    return eta


def calculadora_trabajo(F_txt, x0, x1, E_in, unidad):
    try:
        F = parsear(F_txt)
        W, como = trabajo(F, x0, x1, con_metodo=True)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; {COMO[como]})")
    print(f"W = {muestra(W)} J" + (f" = {cifras(float(W) / 1000, CIFRAS)} kJ" if abs(float(W)) >= 1000 else "")
          + ("   (negativo porque x1 < x0: recorres el intervalo hacia atrás)" if x1 < x0 else ""))
    if E_in:
        try:
            E = a_joules(E_in, unidad)
            eta = eficiencia(W, E)
        except ValueError as err:
            print("Con la energía de entrada:", err)
        else:
            print(f"energía de entrada = {cifras(E, CIFRAS)} J   →   η = {cifras(eta, 4)} = {cifras(100 * eta, 3)} %"
                  "  (η con 4 cifras, porcentaje con 3)")
    try:
        xs = np.linspace(float(x0), float(x1), 400)
        ys = para_graficar(F, xs)
        fig, ax = plt.subplots(figsize=(5.5, 3))
        ax.fill_between(xs, 0, ys, color="0.85"); ax.plot(xs, ys, color="black")
        rotula(ax, xs[len(xs) // 2], ys[len(ys) // 2], "F(x)", xytext=(4, 8))
        ax.set_xlabel("x (m)"); ax.set_ylabel("F (N)"); ax.set_ylim(bottom=min(0, float(np.nanmin(ys))))
        ax.set_title("el área gris es el trabajo W", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_trabajo,
    F_txt=widgets.Text(value="400*9.81 + 1.5*9.81*(30 - y)", description="F =", continuous_update=False),
    x0=widgets.FloatText(value=0, description="x0 (m)"),
    x1=widgets.FloatText(value=30, description="x1 (m)"),
    E_in=widgets.FloatText(value=0.05, description="entrada"),
    unidad=widgets.Dropdown(options=["J", "kWh"], value="kWh", description="unidad"));

In [ ]:
# Casos de prueba de la sección 6.7 (resultado conocido)
_g = sp.Rational("9.81")
_malacate = 400 * _g + sp.Rational("1.5") * _g * (30 - x)

PRUEBAS_7 = [
    ("resorte k = 200 N/m de 0 a 0.1 m: 1 J", lambda: trabajo(200 * x, 0, 0.1) == 1),
    ("resorte k = 800 N/m de 0.05 a 0.15 m: 8 J", lambda: trabajo(800 * x, 0.05, 0.15) == 8),
    ("malacate, 400 kg y cable de 1.5 kg/m, 30 m: 124341.75 J",
     lambda: trabajo(_malacate, 0, 30) == sp.Rational("124341.75")),
    ("malacate con 0.05 kWh de entrada: η = 0.6908",
     lambda: cifras(eficiencia(trabajo(_malacate, 0, 30), a_joules(0.05, "kWh")), 4) == "0.6908"),
    ("bombeo de un tanque de radio 1 m con 3 m de agua hasta el borde: 138685.6 J",
     lambda: round(float(trabajo(1000 * _g * sp.pi * (3 - x), 0, 3)), 1) == 138685.6),
    ("η > 1 se rechaza (violaría la conservación de la energía)", lambda: _rechaza(lambda: eficiencia(150, 100))),
    ("x0 = x1 se rechaza", lambda: _rechaza(lambda: trabajo(5 * x, 1, 1))),
]
for nombre, prueba in PRUEBAS_7:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.7).** Calcula a mano el trabajo para estirar un resorte de $k=500$ N/m desde su longitud natural ($x=0$) hasta $x=0.2$ m y escribe el número en J.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = trabajo(500 * x, 0, sp.Rational(1, 5))
        print("La calculadora da: W =", muestra(ref), "J")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: W = ∫_0^0.2 500x dx = ½ k x²; revisa que elevaste 0.2 al cuadrado")

## 6.8 Fuerza hidrostática y caudal

La presión del agua a la profundidad $h$ es $p=\rho g h$. Sobre una placa vertical cuyo ancho a la profundidad $h$ es $w(h)$, entre $h_1$ y $h_2$:

$$F=\int_{h_1}^{h_2}\rho g\,h\,w(h)\,dh,\qquad h_p=\frac1F\int_{h_1}^{h_2}h\cdot\rho g\,h\,w(h)\,dh\quad(\text{profundidad del centro de presión}).$$

El centro de presión queda más abajo que el centroide de la placa, porque abajo empuja más. En un tubo circular de radio $R$ con perfil de velocidad $v(r)$, cada anillo de radio $r$ y ancho $dr$ tiene área $2\pi r\,dr$:

$$Q=\int_0^Rv(r)\,2\pi r\,dr\quad(\text{m}^3/\text{s}),\qquad \bar v=\frac{Q}{\pi R^2}.$$

El ejemplo exacto es una compuerta rectangular de 2 m de ancho entre 1 m y 2.5 m de profundidad, con $\rho=1000$ kg/m³ y $g=9.81$ m/s².

In [ ]:
rho_g = 1000 * sp.Rational("9.81")
F_ej = sp.integrate(rho_g * x * 2, (x, 1, sp.Rational(5, 2)))
hp_ej = sp.integrate(x * rho_g * x * 2, (x, 1, sp.Rational(5, 2))) / F_ej
print("F =", F_ej, "N ≈", cifras(F_ej, 6), "N   h_p =", hp_ej, "m ≈", cifras(hp_ej, 7), "m")

**Calculadora de la fuerza sobre una placa vertical.** Escribe el ancho $w(h)$ en m como función de la profundidad $h$ (constante para un rectángulo, `h` para un triángulo con el vértice arriba), las profundidades del borde superior y del inferior y la densidad del líquido. La calculadora da la fuerza, el centroide y el centro de presión, y dibuja la placa con la profundidad hacia abajo.

In [ ]:
def fuerza_placa(w, h1, h2, rho=1000, g=9.81):
    """Fuerza hidrostática (N) sobre una placa vertical de ancho w(h) entre las profundidades h1 < h2 (m), y la
    profundidad h_p (m) del centro de presión. También devuelve el área A y la profundidad del centroide h_c."""
    w = sp.sympify(w)
    h1, h2 = exacto(h1, "h1"), exacto(h2, "h2")
    rho, g = exacto(rho, "ρ"), exacto(g, "g")
    if h1 < 0:
        raise ValueError("h1 es una profundidad bajo la superficie: no puede ser negativa.")
    if not h1 < h2:
        raise ValueError("el borde inferior (h2) debe estar más hondo que el superior (h1).")
    if rho <= 0 or g <= 0:
        raise ValueError("ρ y g deben ser positivas.")
    revisa_intervalo(w, h1, h2, "w", var="h")
    anchos = numerica(w)(malla_interior(h1, h2))
    if np.any(anchos < -1e-12):
        raise ValueError("el ancho w(h) sale negativo en parte del intervalo: revisa la fórmula.")
    integ = Integrador()
    def _int(f):
        return integ(sp.expand(f), h1, h2)
    A = _int(w)
    if float(A) <= 0:
        raise ValueError("la placa tiene área cero.")
    F = simplifica(rho * g * _int(x * w))
    hp = simplifica(rho * g * _int(x**2 * w) / F)
    hc = simplifica(_int(x * w) / A)
    return {"F": F, "h_p": hp, "A": A, "h_c": hc, "metodo": integ.metodo}


def calculadora_placa(w_txt, h1, h2, rho):
    try:
        w = parsear(w_txt)
        r = fuerza_placa(w, h1, h2, rho)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; g = 9.81 m/s²; {COMO[r['metodo']]})")
    print(f"F   = {muestra(r['F'])} N = {cifras(float(r['F']) / 1000, CIFRAS)} kN")
    print(f"A   = {muestra(r['A'])} m²    centroide a h_c = {muestra(r['h_c'])} m")
    print(f"centro de presión a h_p = {muestra(r['h_p'])} m de profundidad")
    try:
        hs = np.linspace(float(h1), float(h2), 300)
        ws = para_graficar(w, hs)
        wmax = float(np.nanmax(ws))
        fig, ax = plt.subplots(figsize=(4.8, 3.4))
        ax.fill_betweenx(hs, -ws / 2, ws / 2, facecolor="0.85", edgecolor="black")
        ax.axhline(0, color="black", lw=1.2); rotula(ax, -1.5 * wmax, 0, "superficie", va="bottom", xytext=(0, 3))
        for prof, estilo, texto, lado in ((float(r["h_c"]), ":", "centroide", -1),
                                          (float(r["h_p"]), "--", "centro de presión", 1)):
            ax.axhline(prof, color="black", ls=estilo, lw=1)
            rotula(ax, lado * wmax / 2, prof, f"{texto}, {cifras(prof, CIFRAS)} m", xytext=(lado * 6, 3 * lado),
                   ha="left" if lado > 0 else "right", va="top" if lado > 0 else "bottom")
        ax.set_ylim(float(h2) * 1.08, -0.1 * float(h2)); ax.set_xlim(-1.6 * wmax, 1.6 * wmax)
        ax.set_xlabel("ancho (m)"); ax.set_ylabel("profundidad h (m)"); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_placa,
    w_txt=widgets.Text(value="2", description="w(h) =", continuous_update=False),
    h1=widgets.FloatText(value=1, description="h1 (m)"),
    h2=widgets.FloatText(value=2.5, description="h2 (m)"),
    rho=widgets.FloatText(value=1000, description="ρ (kg/m³)"));

**Calculadora del caudal en un tubo.** Escribe el perfil de velocidad $v(r)$ en m/s con la variable `r` y el radio $R$ del tubo en m. El perfil del ejemplo es el laminar, $v=v_{\max}\left(1-r^2/R^2\right)$ con $v_{\max}=2$ m/s; prueba también uno turbulento, `(1 - r/0.05)^(1/7)`. La calculadora da el caudal y la velocidad media y la compara con la máxima.

In [ ]:
def caudal(v, R):
    """Caudal Q = ∫_0^R v(r) 2πr dr (m³/s) y velocidad media Q/(πR²) (m/s) en un tubo circular de radio R (m)."""
    v = sp.sympify(v)
    R = exacto(R, "R")
    if R <= 0:
        raise ValueError("el radio R debe ser positivo.")
    revisa_intervalo(v, 0, R, "v", var="r")
    rs = np.linspace(0, float(R), 403)
    vs = numerica(v)(rs)
    if not np.all(np.isfinite(vs)):
        raise ValueError("la velocidad debe ser finita en todo el tubo, también en el centro (r = 0) y en la pared.")
    if np.any(vs < -1e-12):
        raise ValueError(f"v(r) sale negativa cerca de r ≈ {cifras(rs[int(np.argmin(vs))], 3)} m: revisa que el radio "
                         "que usaste en la fórmula coincida con R.")
    Q, como = integra(sp.expand(2 * sp.pi * x * v), 0, R)
    Q = simplifica(Q)
    if float(Q) <= 0:
        raise ValueError("el caudal es cero: con v(r) = 0 en todo el tubo no hay flujo.")
    return {"Q": Q, "v_media": simplifica(Q / (sp.pi * R**2)), "metodo": como}


def calculadora_caudal(v_txt, R):
    try:
        v = parsear(v_txt)
        r = caudal(v, R)
        rs = np.linspace(0, R, 300)
        vs = numerica(v)(rs)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; {COMO[r['metodo']]})")
    print(f"Q = {muestra(r['Q'])} m³/s = {cifras(1000 * float(r['Q']), CIFRAS)} L/s")
    print(f"velocidad media = {muestra(r['v_media'])} m/s   máxima = {cifras(float(np.max(vs)), CIFRAS)} m/s   "
          f"media/máxima = {cifras(float(r['v_media']) / float(np.max(vs)), 4)}  (cociente con 4 cifras)")
    try:
        fig, ax = plt.subplots(figsize=(4.8, 3.2))
        rr = np.concatenate((-rs[::-1], rs)); vv = np.concatenate((vs[::-1], vs))
        ax.plot(vv, rr, color="black"); ax.fill_betweenx(rr, 0, vv, color="0.88")
        vm = float(r["v_media"])
        ax.axvline(vm, color="black", ls="--", lw=1); rotula(ax, vm, -0.8 * R, f"media {cifras(vm, CIFRAS)} m/s")
        ax.axhline(R, color="black", lw=2); ax.axhline(-R, color="black", lw=2)
        ax.set_xlabel("v (m/s)"); ax.set_ylabel("r (m)"); ax.set_title("perfil de velocidad (pared en ±R)", fontsize=9)
        plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_caudal,
    v_txt=widgets.Text(value="2*(1 - r^2/0.05^2)", description="v(r) =", continuous_update=False),
    R=widgets.FloatText(value=0.05, description="R (m)"));

In [ ]:
# Casos de prueba de la sección 6.8 (resultado conocido)
PRUEBAS_8 = [
    ("compuerta de 2 m de ancho entre 1 y 2.5 m: F = 51502.5 N y h_p = 1.857143 m",
     lambda: (lambda r: r["F"] == sp.Rational("51502.5") and cifras(r["h_p"], 7) == "1.857143")(fuerza_placa(2, 1, 2.5))),
    ("placa circular de radio 1 m con el centro a 1 m: F = ρ g π = 9810π N y h_p = 1 + (π/4)/π = 1.25 m",
     lambda: (lambda r: sp.simplify(r["F"] - 9810 * sp.pi) == 0 and r["h_p"] == sp.Rational(5, 4))(
         fuerza_placa(2 * sp.sqrt(1 - (x - 1)**2), 0, 2))),
    ("placa cuadrada de 2 m con el borde en la superficie: 39240 N y h_p = 4/3 m",
     lambda: (lambda r: r["F"] == 39240 and r["h_p"] == sp.Rational(4, 3))(fuerza_placa(2, 0, 2))),
    ("placa triangular w = h de 0 a 3 m: 88290 N", lambda: fuerza_placa(x, 0, 3)["F"] == 88290),
    ("borde inferior arriba del superior se rechaza", lambda: _rechaza(lambda: fuerza_placa(1, 2, 1))),
    ("perfil laminar 2(1 - r²/R²), R = 0.05 m: velocidad media 1 m/s",
     lambda: caudal(2 * (1 - x**2 / sp.Rational(1, 20)**2), 0.05)["v_media"] == 1),
    ("perfil (1 - r/R)^(1/7), R = 0.1 m: velocidad media 49/60 de la máxima",
     lambda: caudal((1 - x / sp.Rational(1, 10))**sp.Rational(1, 7), 0.1)["v_media"] == sp.Rational(49, 60)),
    ("perfil uniforme v = 3 m/s: velocidad media 3 m/s", lambda: caudal(3, 0.2)["v_media"] == 3),
    ("radio negativo se rechaza", lambda: _rechaza(lambda: caudal(3, -0.1))),
    ("perfil con velocidad negativa (radio de la fórmula distinto de R) se rechaza",
     lambda: _rechaza(lambda: caudal(2 * (1 - x**2 / sp.Rational(1, 20)**2), 0.1))),
]
for nombre, prueba in PRUEBAS_8:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.8).** Una placa cuadrada de 1 m de lado está vertical en agua, con el borde superior en la superficie. Calcula a mano la fuerza del agua sobre una cara ($\rho=1000$ kg/m³, $g=9.81$ m/s²) y escribe el número en N.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = fuerza_placa(1, 0, 1)["F"]
        print("La calculadora da: F =", muestra(ref), "N")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: F = ∫_0^1 ρ g h · 1 dh = ρ g/2; la presión crece con la profundidad, no es ρ g · 1 en toda la placa")

## 6.9 Carga, energía y valor eficaz de una señal

$$q=\int_{t_0}^{t_1}i(t)\,dt\ \ (\text{C}=\text{A·s}),\qquad E=\int_{t_0}^{t_1}v(t)\,i(t)\,dt\ \ (\text{J}),\qquad f_{\text{ef}}=\sqrt{\frac1T\int_0^Tf(t)^2\,dt}.$$

El valor eficaz (RMS) es el valor constante que disiparía la misma potencia en una resistencia. Para un sensor que despierta, mide, transmite y vuelve a dormir cada periodo $T_p$, la corriente media y la vida de la batería son

$$\bar I=\frac{q_{\text{activo}}+i_{\text{reposo}}\,(T_p-t_{\text{activo}})}{T_p},\qquad \text{vida}=\frac{\text{capacidad}\cdot\text{fracción utilizable}}{\bar I},\qquad 1\ \text{mAh}=3.6\ \text{C}.$$

El ejemplo exacto integra la corriente de arranque de un sensor, $i(t)=3+9e^{-t/0.4}$ mA, durante 1.5 s.

In [ ]:
t_c = sp.symbols("t", nonnegative=True)
q_ej = sp.integrate(3 + 9 * sp.exp(-t_c / sp.Rational(2, 5)), (t_c, 0, sp.Rational(3, 2)))
print("q =", q_ej, "mC ≈", cifras(q_ej, 7), "mC")

**Calculadora de carga y energía.** Escribe $i(t)$ y elige si está en A o en mA; escribe también el voltaje $v(t)$ en V (puede ser constante, o vacío si no lo necesitas) y el intervalo en s (los extremos aceptan fracciones como `1/60`). La calculadora da la carga, la corriente media, la corriente eficaz y la energía entregada.

In [ ]:
def carga(i, t0, t1, con_metodo=False):
    """q = ∫_{t0}^{t1} i(t) dt: en C si i está en A (en mC si está en mA). Con con_metodo=True devuelve (q, método)."""
    i = sp.sympify(i)
    t0, t1 = exacto(t0, "t0"), exacto(t1, "t1")
    if not t0 < t1:
        raise ValueError("t0 debe ser menor que t1.")
    revisa_intervalo(i, t0, t1, "i", var="t")
    q, como = integra(i, t0, t1)
    return (q, como) if con_metodo else q


def energia(v, i, t0, t1):
    """E = ∫_{t0}^{t1} v(t) i(t) dt: en J si v está en V e i en A."""
    return carga(sp.sympify(v) * sp.sympify(i), t0, t1)


def valor_eficaz(f, T, t0=0):
    """Valor eficaz (RMS) de f en un periodo [t0, t0 + T]: √((1/T) ∫ f² dt)."""
    f = sp.sympify(f)
    T, t0 = exacto(T, "T"), exacto(t0, "t0")
    if T <= 0:
        raise ValueError("el periodo T debe ser positivo.")
    return simplifica(sp.sqrt(carga(f**2, t0, t0 + T) / T))


def vida_bateria(q_activo, t_activo, i_reposo, periodo, capacidad_mAh, margen):
    """Corriente media (A) y vida de la batería (años de 365 días) para un ciclo que gasta q_activo (C) durante
    t_activo (s) y luego i_reposo (A) hasta completar el periodo (s). margen es la fracción utilizable (0.8 = 80 %)."""
    q_activo, t_activo, i_reposo = float(q_activo), float(t_activo), float(i_reposo)
    periodo, capacidad_mAh, margen = float(periodo), float(capacidad_mAh), float(margen)
    if q_activo < 0 or i_reposo < 0:
        raise ValueError("la carga activa y la corriente de reposo no pueden ser negativas.")
    if periodo <= 0 or not 0 <= t_activo <= periodo:
        raise ValueError("el periodo debe ser positivo y el tiempo activo debe caber dentro de él.")
    if capacidad_mAh <= 0:
        raise ValueError("la capacidad debe ser positiva.")
    if not 0 < margen <= 1:
        raise ValueError("el margen es la fracción utilizable de la capacidad: un número entre 0 y 1 (0.8 = 80 %).")
    I_media = (q_activo + i_reposo * (periodo - t_activo)) / periodo
    if I_media <= 0:
        raise ValueError("la corriente media es cero: la batería no se gastaría.")
    vida_s = capacidad_mAh * 3.6 * margen / I_media
    return I_media, vida_s / (365 * 24 * 3600)


def calculadora_carga(i_txt, unidad, v_txt, t0_txt, t1_txt):
    try:
        i = parsear(i_txt)
        t0, t1 = exacto(t0_txt, "t0"), exacto(t1_txt, "t1")
        q, como = carga(i, t0, t1, con_metodo=True)
        T = float(t1 - t0)
        ts = np.linspace(float(t0), float(t1), 500)
        ys = para_graficar(i, ts)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    uq, ue = ("C", "J") if unidad == "A" else ("mC", "mJ")
    finitos = ys[np.isfinite(ys)]
    escala = float(np.max(np.abs(finitos))) * T if finitos.size else abs(float(q))
    nula = abs(float(q)) <= 1e-12 * escala            # cancelación numérica: q es 0
    print(f"({CIFRAS} cifras significativas, redondeadas; t en s; {COMO[como]})")
    print(f"carga q = {'0' if nula else muestra(q)} {uq}")
    print(f"corriente media = {'0' if nula else cifras(float(q) / T, CIFRAS)} {unidad}")
    try:
        print(f"corriente eficaz = {cifras(valor_eficaz(i, t1 - t0, t0), CIFRAS)} {unidad}")
    except ERRORES as err:
        print("corriente eficaz: no existe, porque la integral de i² diverge" if "diverge" in str(err)
              else f"corriente eficaz: {explica(err)}")
    if v_txt.strip():
        try:
            print(f"energía E = {muestra(energia(parsear(v_txt), i, t0, t1))} {ue}")
        except ERRORES as err:
            print("energía:", explica(err))
    try:
        fig, ax = plt.subplots(figsize=(5.5, 3))
        ax.fill_between(ts, 0, ys, color="0.85"); ax.plot(ts, ys, color="black")
        im = 0.0 if nula else float(q) / T
        ax.axhline(im, color="black", ls="--", lw=1); rotula(ax, ts[-1], im, "media", xytext=(-30, 7))
        ax.set_xlabel("t (s)"); ax.set_ylabel(f"i ({unidad})"); ax.set_ylim(bottom=min(0, float(np.nanmin(ys))))
        ax.set_title(f"el área gris es la carga q, en {uq}", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


widgets.interact(calculadora_carga,
    i_txt=widgets.Text(value="3 + 9*exp(-t/0.4)", description="i(t) =", continuous_update=False),
    unidad=widgets.Dropdown(options=["A", "mA"], value="mA", description="i en"),
    v_txt=widgets.Text(value="3.3", description="v(t) (V) =", continuous_update=False),
    t0_txt=widgets.Text(value="0", description="t0 (s)", continuous_update=False),
    t1_txt=widgets.Text(value="1.5", description="t1 (s)", continuous_update=False));

**Calculadora de la vida de una batería.** Escribe la carga que gasta el sensor en cada ciclo activo (en mC; por ejemplo, los 8.015 mC del arranque más 40 mA durante 0.12 s de transmisión dan 12.815 mC en 1.62 s), la corriente de reposo, el periodo y la batería. La gráfica muestra cómo cambia la vida con el periodo.

In [ ]:
def calculadora_bateria(q_mC, t_activo, i_reposo_uA, periodo, capacidad_mAh, util_pct):
    try:
        I, anios = vida_bateria(q_mC / 1000, t_activo, i_reposo_uA * 1e-6, periodo, capacidad_mAh, util_pct / 100)
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    print(f"({CIFRAS} cifras significativas, redondeadas; años de 365 días)")
    print(f"corriente media = {cifras(I * 1e6, CIFRAS)} µA")
    print(f"vida = {cifras(anios, CIFRAS)} años = {cifras(anios * 365, CIFRAS)} días")
    try:
        periodos = np.linspace(max(t_activo * 1.05, 0.1 * periodo), 5 * periodo, 200)
        vidas = [vida_bateria(q_mC / 1000, t_activo, i_reposo_uA * 1e-6, p, capacidad_mAh, util_pct / 100)[1]
                 for p in periodos]
        fig, ax = plt.subplots(figsize=(5.5, 3))
        ax.plot(periodos, vidas, color="black"); ax.plot([periodo], [anios], marker="o", color="black")
        rotula(ax, periodo, anios, f"tu diseño: {cifras(anios, CIFRAS)} años", xytext=(6, -10))
        ax.set_xlabel("periodo entre mediciones (s)"); ax.set_ylabel("vida (años)"); ax.set_ylim(bottom=0)
        ax.set_title("vida de la batería contra el periodo", fontsize=9); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


_ancho = {"description_width": "initial"}
widgets.interact(calculadora_bateria,
    q_mC=widgets.FloatText(value=12.815336, description="carga activa (mC)", style=_ancho),
    t_activo=widgets.FloatText(value=1.62, description="tiempo activo (s)", style=_ancho),
    i_reposo_uA=widgets.FloatText(value=6, description="corriente de reposo (µA)", style=_ancho),
    periodo=widgets.FloatText(value=120, description="periodo (s)", style=_ancho),
    capacidad_mAh=widgets.FloatText(value=2600, description="capacidad (mAh)", style=_ancho),
    util_pct=widgets.IntSlider(value=80, min=10, max=100, description="utilizable (%)", style=_ancho,
                               continuous_update=False));

In [ ]:
# Casos de prueba de la sección 6.9 (resultado conocido)
_pwm = sp.Piecewise((1, x < sp.Rational(1, 4)), (0, True))

PRUEBAS_9 = [
    ("i = 2 A de 0 a 5 s: 10 C", lambda: carga(2, 0, 5) == 10),
    ("i = 3 + 9e^(-t/0.4) mA de 0 a 1.5 s: 8.015336 mC",
     lambda: cifras(carga(3 + 9 * sp.exp(-x / sp.Rational(2, 5)), 0, 1.5), 7) == "8.015336"),
    ("i = 0.5e^(-50t) A de 0 a 0.05 s: 0.0091792 C",
     lambda: cifras(carga(sp.exp(-50 * x) / 2, 0, 0.05), 5) == "0.0091792"),
    ("v = 3.3 V e i = 2 A durante 5 s: 33 J", lambda: energia(sp.Rational("3.3"), 2, 0, 5) == 33),
    ("valor eficaz de la sierra v = t en [0, 1): 1/√3",
     lambda: sp.simplify(valor_eficaz(x, 1) - 1 / sp.sqrt(3)) == 0),
    ("valor eficaz de un PWM de amplitud 1 y ciclo de trabajo 0.25: 0.5", lambda: valor_eficaz(_pwm, 1) == sp.Rational(1, 2)),
    ("el mismo PWM escrito con escalón: 0.5", lambda: valor_eficaz(parsear("escalon(0.25 - t)"), 1) == sp.Rational(1, 2)),
    ("vida del sensor (12.815336 mC en 1.62 s, 6 µA, 120 s, 2600 mAh, 80 %): 112.71 µA y 2.1066 años",
     lambda: (lambda r: cifras(r[0] * 1e6, 5) == "112.71" and cifras(r[1], 5) == "2.1066")(
         vida_bateria(12.815336e-3, 1.62, 6e-6, 120, 2600, 0.8))),
    ("3.6 C cada hora, sin reposo (1 mA de media) con 2600 mAh al 100 %: 2600 h",
     lambda: (lambda r: cerca(r[0], 1e-3, 1e-15) and cerca(r[1], 2600 / 8760, 1e-12))(
         vida_bateria(3.6, 3600, 0, 3600, 2600, 1))),
    ("1 mA constante con 1000 mAh: 1000 h = 1000/8760 años",
     lambda: cerca(vida_bateria(0, 0, 1e-3, 1, 1000, 1)[1], 1000 / 8760, 1e-12)),
    ("margen mayor que 1 se rechaza", lambda: _rechaza(lambda: vida_bateria(0.01, 1, 6e-6, 120, 2600, 80))),
    ("tiempo activo mayor que el periodo se rechaza", lambda: _rechaza(lambda: vida_bateria(0.01, 200, 6e-6, 120, 2600, 0.8))),
]
for nombre, prueba in PRUEBAS_9:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.9).** Un capacitor se descarga con $i(t)=0.5\,e^{-50t}$ A. Calcula a mano la carga que pasa entre $t=0$ y $t=0.1$ s y escribe el número en C.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = carga(sp.exp(-50 * x) / 2, 0, sp.Rational(1, 10))
        print("La calculadora da: q =", muestra(ref), "C")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: la antiderivada de 0.5 e^(-50t) es -0.01 e^(-50t); evalúa entre 0 y 0.1")

## 6.10 Costo acumulado

$$C(q_1)-C(q_0)=\int_{q_0}^{q_1}C'(q)\,dq,\qquad C(T)=P+\int_0^Tm(t)\,dt.$$

El costo marginal $C'(q)$ es lo que cuesta producir la pieza número $q$; integrarlo da el costo de un lote. Para una máquina con precio de compra $P$ y tasa de mantenimiento $m(t)$ en pesos por año, $C(T)$ es lo que llevas gastado a los $T$ años. Entre dos máquinas, la más cara al comprar puede salir más barata con el tiempo si su mantenimiento crece más despacio; el punto de equilibrio es el $T>0$ donde $C_A(T)=C_B(T)$.

En el ejemplo exacto, la máquina A cuesta 250 000 pesos con $m_A=20\,000+6\,000t$ y la B cuesta 400 000 pesos con $m_B=12\,000+1\,500t$.

In [ ]:
T_s, t_m = sp.symbols("T t", nonnegative=True)
C_A = 250000 + sp.integrate(20000 + 6000 * t_m, (t_m, 0, T_s))
C_B = 400000 + sp.integrate(12000 + 1500 * t_m, (t_m, 0, T_s))
print("C_A(T) =", C_A, "   C_B(T) =", C_B)
print("equilibrio:", sp.solveset(C_A - C_B, T_s, sp.Interval.open(0, sp.oo)), "años")

**Calculadora de equilibrio entre dos máquinas.** Escribe el precio de compra de cada máquina en pesos y su tasa de mantenimiento $m(t)$ en pesos por año, con $t$ en años. La calculadora busca los tiempos positivos en que los costos acumulados se igualan (exactos con `sympy` si se puede; si no, numéricos dentro del horizonte) y dibuja $C_A(T)$ y $C_B(T)$.

In [ ]:
_TAU10 = sp.Dummy("tau", real=True)


def costo_total(C_marginal, q0, q1, fijo=0):
    """Costo de producir de q0 a q1: fijo + ∫_{q0}^{q1} C'(q) dq (en las unidades de C' por unidad de q)."""
    Cm = sp.sympify(C_marginal)
    q0, q1, fijo = exacto(q0, "q0"), exacto(q1, "q1"), exacto(fijo, "el costo fijo")
    if q0 < 0 or not q0 < q1:
        raise ValueError("necesitas 0 ≤ q0 < q1.")
    revisa_intervalo(Cm, q0, q1, "C'", var="q")
    return simplifica(fijo + integra(Cm, q0, q1)[0])


def costo_acumulado(precio, m, T):
    """C(T) = precio + ∫_0^T m(t) dt: lo gastado en una máquina a los T años (m en pesos por año)."""
    precio, T = exacto(precio, "el precio"), exacto(T, "T")
    if T < 0:
        raise ValueError("T es un tiempo: no puede ser negativo.")
    if T == 0:
        return precio
    m = sp.sympify(m)
    revisa_intervalo(m, 0, T, "m", var="t")
    return simplifica(precio + integra(m, 0, T)[0])


def _acumulada(m):
    """∫_0^x m(t) dt como expresión en x (para resolver C_A = C_B), o None si sympy no da una fórmula útil."""
    m = sp.sympify(m)
    if m.has(sp.Heaviside):
        m = m.rewrite(sp.Piecewise)
    try:
        r = con_limite(lambda: sp.integrate(m.subs(x, _TAU10), (_TAU10, 0, x)), 10)
    except (ValueError, NotImplementedError, TypeError, ArithmeticError):
        return None
    if r.has(*_RARAS) or r.has(sp.I, sp.oo, -sp.oo, sp.zoo, sp.nan):
        return None
    return r


def _acumula_malla(f, ts):
    """∫_0^t f en cada punto de la malla, con la regla del punto medio (no evalúa f en t = 0, donde puede no existir)."""
    v = f((ts[1:] + ts[:-1]) / 2)
    return np.concatenate(([0.0], np.cumsum(np.diff(ts) * v)))


def equilibrio(precio_A, m_A, precio_B, m_B, horizonte=100, con_metodo=False):
    """Tiempos T > 0 (años) en que C_A(T) = C_B(T), en orden. Exactos con sympy si se puede (todos los T > 0);
    si no, se buscan numéricamente solo en (0, horizonte]. Lista vacía si no se igualan. Con con_metodo=True
    devuelve (tiempos, "exacta" o "numérica")."""
    pA, pB = exacto(precio_A, "el precio de A"), exacto(precio_B, "el precio de B")
    m_A, m_B = sp.sympify(m_A), sp.sympify(m_B)
    if not horizonte > 0:
        raise ValueError("el horizonte debe ser positivo.")
    for m, nombre in ((m_A, "m_A"), (m_B, "m_B")):
        revisa_intervalo(m, 0, horizonte, nombre, var="t")
    def _sale(tiempos, metodo):
        return (tiempos, metodo) if con_metodo else tiempos
    acum = _acumulada(m_A - m_B)
    if acum is not None:
        D = (pA - pB) + acum
        if simplifica(D) == 0:
            raise ValueError("las dos máquinas cuestan lo mismo en todo momento: no hay un punto de equilibrio único.")
        try:
            sol = con_limite(lambda: sp.solveset(D, x, sp.Interval.open(0, sp.oo)), 10)
        except (ValueError, NotImplementedError, TypeError):
            sol = None
        if sol == sp.EmptySet:
            return _sale([], "exacta")
        if isinstance(sol, sp.FiniteSet) and all(c.is_real for c in sol):
            return _sale(sorted((simplifica(c) for c in sol), key=float), "exacta")
    fd = numerica(m_A - m_B)                        # numérico: cambios de signo de C_A - C_B en una malla
    ts = np.linspace(0, float(horizonte), 4001)
    D = float(pA - pB) + _acumula_malla(fd, ts)
    def D_exacta(t):
        return float(pA - pB) + quad(lambda s_: float(fd(s_)), 0, t, limit=200)[0]
    raices = []
    for k in range(1, len(ts) - 1):
        if D[k] == 0:
            raices.append(ts[k])
        elif D[k] * D[k + 1] < 0:
            try:
                raices.append(brentq(D_exacta, ts[k], ts[k + 1], xtol=1e-12))
            except ValueError:                      # quad y la malla difieren en el signo: interpolación lineal
                raices.append(ts[k] - D[k] * (ts[k + 1] - ts[k]) / (D[k + 1] - D[k]))
    return _sale([sp.Float(r, 12) for r in raices], "numérica")


def calculadora_equilibrio(precio_A, m_A_txt, precio_B, m_B_txt, horizonte):
    try:
        m_A, m_B = parsear(m_A_txt), parsear(m_B_txt)
        if precio_A < 0 or precio_B < 0:
            raise ValueError("los precios no pueden ser negativos.")
        tiempos, metodo = equilibrio(precio_A, m_A, precio_B, m_B, horizonte, con_metodo=True)
        costos = [costo_acumulado(precio_A, m_A, T) for T in tiempos]
    except ERRORES as err:
        print("Revisa la entrada:", explica(err)); return
    exacta = metodo == "exacta"
    print(f"({CIFRAS} cifras significativas, redondeadas; T en años, costos en pesos; "
          + ("cálculo exacto con sympy, para todo T > 0)" if exacta else
             f"búsqueda numérica, solo en (0, {horizonte}] años)"))
    if not tiempos:
        print("los costos acumulados no se igualan para ningún T > 0" if exacta else
              f"los costos acumulados no se igualan en (0, {horizonte}] años; prueba un horizonte mayor")
    for T, C in zip(tiempos, costos):
        print(f"equilibrio en T = {muestra(T)} años: los dos costos llegan a {cifras(C, CIFRAS)} pesos"
              + ("   (fuera del horizonte de la gráfica)" if float(T) > horizonte else ""))
    try:
        def _barata(T):
            dif = float(costo_acumulado(precio_A, m_A, T) - costo_acumulado(precio_B, m_B, T))
            return "A" if dif < 0 else ("B" if dif > 0 else "ninguna")
        if tiempos:
            print(f"antes del primer equilibrio sale más barata la máquina {_barata(float(tiempos[0]) / 2)}; "
                  f"después del último, la {_barata(float(tiempos[-1]) + 1)}")
        Ts = np.linspace(0, horizonte, 600)
        CA = precio_A + _acumula_malla(numerica(m_A), Ts)
        CB = precio_B + _acumula_malla(numerica(m_B), Ts)
        fig, ax = plt.subplots(figsize=(5.5, 3.2))
        ax.plot(Ts, CA, color="black", ls="-"); ax.plot(Ts, CB, color="black", ls="--")
        rotula(ax, Ts[-1], CA[-1], "A"); rotula(ax, Ts[-1], CB[-1], "B")
        for T, C in zip(tiempos, costos):
            if float(T) <= horizonte:
                ax.axvline(float(T), color="0.4", ls=":", lw=1)
                rotula(ax, float(T), float(C), f"T = {cifras(T, CIFRAS)} años", xytext=(-6, 12), ha="right")
        ax.set_xlabel("T (años)"); ax.set_ylabel("costo acumulado (pesos)")
        ax.ticklabel_format(axis="y", style="plain"); plt.show()
    except ERRORES:
        print("(no pude dibujar la gráfica con estos datos)")


_ancho = {"description_width": "initial"}
widgets.interact(calculadora_equilibrio,
    precio_A=widgets.FloatText(value=250000, description="precio A (pesos)", style=_ancho),
    m_A_txt=widgets.Text(value="20000 + 6000*t", description="m_A(t) (pesos/año) =", continuous_update=False, style=_ancho),
    precio_B=widgets.FloatText(value=400000, description="precio B (pesos)", style=_ancho),
    m_B_txt=widgets.Text(value="12000 + 1500*t", description="m_B(t) (pesos/año) =", continuous_update=False, style=_ancho),
    horizonte=widgets.IntSlider(value=12, min=1, max=30, description="horizonte (años)", style=_ancho,
                                continuous_update=False));

In [ ]:
# Casos de prueba de la sección 6.10 (resultado conocido)
PRUEBAS_10 = [
    ("C'(q) = 120 - 0.4q de q = 100 a 150: 3500 pesos", lambda: costo_total(120 - sp.Rational(2, 5) * x, 100, 150) == 3500),
    ("máquinas A y B: equilibrio en T = (-16 + 2√1414)/9 ≈ 6.5785 años",
     lambda: (lambda c: len(c) == 1 and sp.simplify(c[0] - (-16 + 2 * sp.sqrt(1414)) / 9) == 0 and cifras(c[0], 5) == "6.5785")(
         equilibrio(250000, 20000 + 6000 * x, 400000, 12000 + 1500 * x))),
    ("C_A(5) = 425000 pesos", lambda: costo_acumulado(250000, 20000 + 6000 * x, 5) == 425000),
    ("C_B(10) = 595000 pesos", lambda: costo_acumulado(400000, 12000 + 1500 * x, 10) == 595000),
    ("curva de aprendizaje 10·q^(-0.32) de 0 a 100 piezas: 336.89 minutos",
     lambda: round(float(costo_total(parsear("10*q^(-0.32)"), 0, 100)), 2) == 336.89),
    ("mismo mantenimiento y distinto precio: nunca se igualan", lambda: equilibrio(100, 1, 200, 1) == []),
    ("q1 < q0 se rechaza", lambda: _rechaza(lambda: costo_total(10, 50, 0))),
]
for nombre, prueba in PRUEBAS_10:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6.10).** Cada pieza cuesta 10 pesos, sin importar cuántas hagas ($C'(q)=10$). Calcula a mano el costo de producir las primeras 50 piezas, sin costo fijo, y escribe el número en pesos.

In [ ]:
mi_valor = None     # escribe un número

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = costo_total(10, 0, 50)
        print("La calculadora da:", muestra(ref), "pesos")
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: ∫_0^50 10 dq es el área de un rectángulo de base 50 y altura 10")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = (PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
         + PRUEBAS_6 + PRUEBAS_7 + PRUEBAS_8 + PRUEBAS_9 + PRUEBAS_10)
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")